# In the Shadow of the Hadamard Test — *From Garbage to Spectra*

### Qiskit Hackathon Challenge · Participant Notebook

---

## The one-paragraph version

The **Hadamard test** is the workhorse circuit of quantum algorithms: one ancilla qubit controls a
unitary $U$ acting on an $n$-qubit system register, and interference on the ancilla reveals
$\operatorname{Tr}[U\rho]$. It is the engine inside phase estimation, overlap estimation and
quantum linear algebra. In every textbook treatment, the ancilla is measured and the system
register — now entangled, scrambled, and generally called the *garbage state* — is discarded.

In 2025 Faehrmann, Eisert and Kueng pointed out that this is a waste. If you also measure the
system register in **random Pauli bases** and keep every shot's record, then the very same circuit
runs deliver, at essentially zero extra cost:

* the usual complex signal $\chi(t) = \operatorname{Tr}[U(t)\rho]$ from the ancilla;
* the input state's **energy** $\langle H\rangle$ and any **conserved charge** $\langle Q\rangle$;
* and — the part with no classical analogue — the **joint observables**
  $\chi_O(t) = \operatorname{Tr}[O\,U(t)\rho]$, obtained by *correlating* the ancilla outcome with
  the system-register classical shadow.

**Your job in this hackathon is to build that machine and then do something useful with it.**

> **Reference.** P. K. Faehrmann, J. Eisert, R. Kueng, *In the Shadow of the Hadamard Test: Using
> the Garbage State for Good and Further Modifications*, **Phys. Rev. Lett. 135, 150603 (2025)**;
> extended version [arXiv:2505.15913](https://arxiv.org/abs/2505.15913). Equation numbers used
> throughout — (D1), (D11), (C1) — refer to that paper's appendices.

---

## Why anyone should care

Standard Hadamard test | Shadow-enhanced Hadamard test
:--|:--
1 circuit family → 1 number, $\chi(t)$ | 1 circuit family → $\chi(t)$, $\langle H\rangle$, $\langle Q\rangle$, and $\chi_O(t)$ for **any** $O$ you think of later
system register discarded | system register is a classical shadow you can re-mine forever
each new observable needs a new experiment | each new observable is a line of NumPy over saved records
extra cost: — | extra cost: up to $2n$ single-qubit gates and $n$ measurements per shot

The flagship consequence you will demonstrate: the ancilla data alone tells you **which energies**
a state populates; the ancilla-correlated garbage register tells you **which symmetry sector each
of those energies lives in**. That is a two-dimensional spectroscopy from a one-dimensional
experiment.

---

## What you must deliver

| | Goal | Required output |
|---|---|---|
| **Part A** | **Fundamental goal — mandatory** | Implement and validate the shadow-enhanced Hadamard test: $\hat\chi(t)$, $\langle H\rangle$, $\langle Q\rangle$, one non-conserved observable, and one complex joint observable $\chi_O(t)$ — all from **one** set of circuits, all agreeing with exact references inside their error bars, with the error visibly falling as $N^{-1/2}$. |
| **Part B** | **Advanced goal — pick exactly ONE track** | **Track A (flagship)**: a symmetry-resolved spectral analyzer reporting $(E_k,\;p_k,\;\hat q_k)$ with uncertainties. Tracks **B–E** are alternative applications — see §7.0. Whatever you pick, your estimator may consume **measurement data only**. |
| **Part C** | **Bonus layers — optional, stack on any track** | A Krylov energy solver from the same records (§8) and a noise-robustness study (§9), both worked through below; measurement optimisation is left open in the Going-further boxes. |

Three rules apply everywhere and are non-negotiable at judging time:

1. **Estimators consume measurement data only.** Exact diagonalisation is for *evaluation*.
   Checkpoint 7, item **7.7**, enforces this mechanically for Track A; your own estimator should
   pass the same tripwire, and saying so in your report is worth doing.
2. **Every number carries an uncertainty.** A point estimate with no error bar is not a result.
3. **Resources are reported.** Circuits, shots, transpiled 1q/2q gate counts, depth, and classical
   post-processing time.

---

## How this notebook is organised

The notebook is a **guided build**. It walks the full pipeline in dependency order, and at each
step there is a numbered **🎯 Challenge** stating the specification. In the participant edition
four of them (2, 5, 6, 11) are already implemented for you — see the worklist in the notice at
the top for what is actually missing. Most challenges
are followed by a **✅ Checkpoint** that grades them with hard `assert`s; the few that are not are
marked in the table.

| # | Challenge | § | What it covers (see the notice above for what you implement) | Graded by |
|---:|---|---|---|---|
| **1** | Controlled time evolution | 3.1 | `build_controlled_evolution` — exact and Trotter paths | Checkpoint 3a |
| **2** | The shadow-Hadamard circuit | 3.2 | `build_shadow_hadamard_circuit` | Checkpoint 3b |
| **3** | Per-shot records | 4 | `parse_memory`, `run_shadow_hadamard`, `ShadowRecords` | Checkpoint 4 |
| **4** | The three estimators | 5 | `pauli_snapshot_values` and the ancilla / unweighted / ancilla-weighted estimators | Checkpoint 5 |
| **5** | The time sweep | 6.1 | `run_time_sweep` — one pass, every quantity, records retained | — |
| **6** | Validation and shot scaling | 6.2–6.4 | the full Part-A acceptance evidence | Checkpoint 6 (8 items) |
| **7** | DFT baseline | 7.1 | `dft_spectrum` + **data-only** peak finding and labels | 2 inline asserts |
| **8** | Matrix-pencil reconstruction | 7.2 | `matrix_pencil`, `amplitudes_at`, `reconstruct` | Checkpoint 7 |
| **9** | Bootstrap uncertainties | 7.3 | `bootstrap_uncertainties` | Checkpoint 7 |
| **10** | *(bonus)* Krylov energy solver | 8 | `krylov_lowest_energy` + a regularisation study | 1 advisory |
| **11** | *(bonus)* Noise robustness | 9 | a noise model and an honest damage report | 1 advisory |

Challenges 1–6 are Part A. Challenges 7–9 are Part B Track A. Challenges 10–11 are Part C.

Beyond the challenges, four **🧪 Going further** boxes list eighteen open extensions that are
*deliberately not implemented here*. Those are where the Originality and Advanced-application
rubric points live — the challenges get you to a working baseline, the Going-further boxes are how
you win.

### Conventions used in this notebook

* **✅ Checkpoint** cells contain `assert`s. Green means that section is provably correct. Set
  `STRICT_CHECKS = False` in §0.2 if you would rather collect failures than stop at the first one.
  Checks whose *tolerance* was calibrated at the official shot budget become advisory when
  `BUDGET = "fast"`; the deterministic ones — conventions, bit ordering, operator identities —
  stay hard at every budget.
* **⚠️ Trap** callouts mark the specific mistakes that have cost people hours. There are two
  different endianness conventions in this project, one published sign typo, and one transpiler
  reordering hazard. All three are flagged where they bite.
* Runtime is about **3 minutes** end to end at the official benchmark budget (2 min 40 s measured
  on an 8-core desktop), of which the main sweep is ≈ 50 s. `BUDGET = "fast"` halves the sweep
  while you debug; the checkpoints and the scaling study cost the same either way, and the noise
  bonus in §9 sub-samples the same time grid, so it halves too.
* Every figure is written to `./figures/` as it is drawn, and the headline numbers are dumped to
  `run_summary.json` by the last cell — both are submission deliverables.

---

## Before you start

On Colab: *Runtime → Run all*. The install cell takes about a minute on a fresh VM; nothing here
needs a GPU. Locally, any Python ≥ 3.10 with the packages listed in §0.1.

Skim §1 (theory) and §3.4 (measurement design) before writing code. §3.4 in particular answers two
questions every team asks in the first hour — *do I need mid-circuit measurement?* and *does Qiskit
have a classical-shadows API?* — and the answers save real time.

Then run the notebook top to bottom once, unmodified, so you have a working baseline and a set of
green checkpoints to regress against. Only then start changing things.

---

> ## 📝 This is the participant edition
>
> Some function bodies below have been replaced by `# TODO` blocks — the **signature, type hints,
> docstring and a set of implementation hints are kept**, and the body ends in
> `raise NotImplementedError`. Everything else is complete and working.
>
> **How to work through it.**
>
> 1. Run the notebook from the top. It will execute cleanly until it hits the first unimplemented
>    function, then stop with a clear `NotImplementedError` naming the challenge.
> 2. Read that challenge's **🎯 box** — it is the complete specification: signatures, return
>    arities, the traps, and which checkpoint grades it.
> 3. Fill in the body, re-run the cell, then run the **✅ Checkpoint** below it.
> 4. Green checkpoint → move on. Red → the failure message tells you which identity broke.
>
> The stubs are ordered by dependency, so working straight down the notebook always works. Nothing
> later secretly depends on something you have not written yet.
>
> **You are done with the guided build** when the last cell prints `56 / 56 checkpoints passed`.
> Two legitimate variations: the Krylov check in §8 is advisory, so a correct run may report
> `55 / 56`; and the optional `SamplerV2` cell in §4.2 is skipped entirely if that import is
> unavailable, giving `55 / 55`.
> That is the floor, not the goal — the 🧪 *Going further* boxes are where the Originality and
> Advanced-application marks live, and none of those are implemented at all.
>
> ### Your actual worklist
>
> **11 function bodies**, in Challenges 1, 3, 4, 7, 8, 9, 10:
>
> | Challenge | you implement |
> |---:|---|
> | 1 | `build_controlled_evolution` |
> | 3 | `parse_memory`, `run_shadow_hadamard` |
> | 4 | `estimate_joint_observable`, `estimate_system_observable`, `pauli_snapshot_values` |
> | 7 | `dft_peak_labels` |
> | 8 | `matrix_pencil`, `reconstruct` |
> | 9 | `bootstrap_uncertainties` |
> | 10 | `krylov_lowest_energy` |
>
> Given complete (read them, do not rewrite them): `ShadowRecords`, `SweepResult`,
> `pauli_terms`, `_check_quadrature_pair`, `estimate_hadamard_signal`, `amplitudes_at`,
> `build_shadow_hadamard_circuit`, `dft_spectrum`, `run_time_sweep`, `simple_noise_model`,
> and every validation, plotting and checkpoint cell. Note in particular that Challenge 3's
> box names `ShadowRecords`, and Challenge 4's box says "build five functions and a guard" —
> three of those six are already written for you.
>
> Everything else is **given, and working** — the Hadamard circuit itself, the time sweep, and all
> the validation, evaluation and plotting code.

# 0 · Setup

## 0.1 Environment

The notebook needs `qiskit ≥ 2.1`, `qiskit-aer`, `numpy`, `scipy`, `matplotlib`, and
`pylatexenc` for the circuit drawings. The cell below checks each one by *import* and installs only
what is actually missing, so it is a no-op on a machine that already has them and takes about a
minute on a fresh Colab VM.

Why check by import rather than just running `pip install`? Because on Colab an unconditional
install of `qiskit` can pull a different minor version than the one already loaded in the kernel,
and you then get import errors that look like your code is broken when it is not. Checking first
avoids that entirely.

**Qiskit ≥ 2.1 is a hard requirement**, asserted in the next cell. Qiskit 2.0 removed `c_if` and
reworked the primitives interface; anything below 2.1 will fail on the `SamplerV2` demonstration in
§4.2 and may behave differently around `PauliEvolutionGate` synthesis.

> **Pinned reference environment:** `qiskit 2.5.1`, `qiskit-aer 0.17.2`, `numpy 2.5.2`,
> `scipy 1.18.0`, Python 3.12. Every number quoted in this notebook was produced with exactly those versions. If
> yours differ, expect the *statistics* to match and the last decimal place not to — that is fine,
> and the checkpoints are built with enough margin to tolerate it. Freeze your own versions in the
> environment file you submit.

In [1]:
# Colab-safe dependency bootstrap. Re-running is harmless.
import importlib, subprocess, sys

def _ensure(requirements):
    missing = []
    for module_name, pip_spec in requirements:
        try:
            importlib.import_module(module_name)
        except ImportError:
            missing.append(pip_spec)
    if missing:
        print("Installing:", " ".join(missing))
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("All dependencies already present.")

_ensure([
    ("qiskit",     "qiskit>=2.1"),
    ("qiskit_aer", "qiskit-aer>=0.15"),
    ("numpy",      "numpy"),
    ("scipy",      "scipy"),
    ("matplotlib", "matplotlib"),
    ("pylatexenc", "pylatexenc"),      # nicer circuit drawings
])

All dependencies already present.


With the dependencies in place, we import nearly everything the notebook uses in one cell so that
later sections stay uncluttered (a few section-local imports appear where they are used), and we assert the Qiskit major version. Three imports are worth
pointing out now because they carry the physics:

* `SparsePauliOp` — the sparse Pauli representation we use for $H$, $Q$ and every observable. It
  is also what makes the shadow estimator a one-liner, because it hands us the Pauli decomposition
  for free.
* `PauliEvolutionGate` + `SuzukiTrotter` — the hardware-realistic route to $e^{-iHt}$.
* `AerSimulator` — the mandatory backend for this challenge, in ideal mode for everything except
  the §9 noise bonus.

In [2]:
from __future__ import annotations

import json
import os
import time
from dataclasses import dataclass
from typing import Iterable, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import qiskit
import qiskit_aer
import scipy
from scipy.linalg import eigvals, hankel, lstsq, svd

from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister, transpile
from qiskit.circuit import Gate
from qiskit.circuit.library import PauliEvolutionGate, UnitaryGate
from qiskit.quantum_info import (
    Operator, SparsePauliOp, Statevector, partial_trace,
)
from qiskit.synthesis import SuzukiTrotter
from qiskit_aer import AerSimulator

print(f"qiskit      {qiskit.__version__}")
print(f"qiskit-aer  {qiskit_aer.__version__}")
print(f"numpy       {np.__version__}")
print(f"scipy       {scipy.__version__}")

_major, _minor = (int(x) for x in qiskit.__version__.split(".")[:2])
assert (_major, _minor) >= (2, 1), "This notebook targets Qiskit >= 2.1 (control-flow / primitives API)."

qiskit      2.0.2
qiskit-aer  0.17.1
numpy       1.26.4
scipy       1.17.1


AssertionError: This notebook targets Qiskit >= 2.1 (control-flow / primitives API).

## 0.2 Experiment budget

Everything downstream reads these three numbers. The **official benchmark budget** (challenge
specification document, §4.3) is `N_TIMES = 64`, `DT = 0.2`, `SHOTS = 2000` — that is what your submission
must use, and it costs about 20–60 s of simulator time. A reduced `"fast"` grid (32 times,
$\Delta t = 0.4$, half the total shots) is provided for quick iteration while you are debugging.

> **Use `"full"` for anything you report.** At the reduced budget the Part-B reconstruction still
> works but the weakest spectral line ($p\approx0.05$) becomes marginal, and the §8 Krylov GEVP —
> which is deliberately ill-conditioned — becomes unreliable. The advanced checkpoints below are
> calibrated for the official budget and are automatically relaxed to *advisory* when
> `BUDGET != "full"`.

Why these numbers?

* $T_{\max} = (N_t-1)\,\Delta t = 12.6$ → Fourier resolution $2\pi/T \approx 0.50$, comfortably
  below the smallest populated level spacing of the benchmark ($1.03$).
* Nyquist window $|E| \le \pi/\Delta t \approx 15.7$, far outside the full spectral radius
  $2.64$ → **no aliasing**.
* Total shots $= N_t \times 2\ \text{quadratures} \times \text{shots} = 256\,000$.

You may re-allocate shots or re-space the grid **only** under the same totals and the same
$T_{\max}$.

In [ ]:
# ----------------------------------------------------------------- experiment budget
BUDGET = "full"        # "full" = official benchmark. "fast" = half the shots, for debugging.

if BUDGET == "full":
    N_TIMES, DT, SHOTS = 64, 0.2, 2000          # official challenge budget: 256,000 shots
elif BUDGET == "fast":
    N_TIMES, DT, SHOTS = 32, 0.4, 2000          # T_max 12.4 (vs 12.6), half the shots, ~2x faster
else:
    raise ValueError(BUDGET)

SEED = 2026            # master seed -- EVERY random draw in this notebook derives from it
STRICT_CHECKS = True   # checkpoints raise on failure (set False to keep going and inspect)

TS = np.arange(N_TIMES) * DT
T_MAX = TS[-1]

print(f"budget            : {BUDGET}")
print(f"times             : N_t = {N_TIMES}, dt = {DT}, T_max = {T_MAX:.2f}")
print(f"shots per (t,phi) : {SHOTS}")
print(f"total shots       : {N_TIMES * 2 * SHOTS:,}")
print(f"Fourier resolution: 2*pi/T_max = {2 * np.pi / T_MAX:.3f}")
print(f"Nyquist window    : |E| <= pi/dt = {np.pi / DT:.2f}")
if BUDGET != "full":
    print("\nNOTE: reduced budget. Checks whose tolerance was calibrated at the official budget")
    print("      become ADVISORY; deterministic checks stay hard. Re-run at 'full' before you")
    print("      believe any number you intend to report.")

Two small pieces of infrastructure that the rest of the notebook leans on.

`style_axes` just keeps the figures consistent — ignore it. The important one is **`check()`**,
which is how every section grades itself:

* it prints `PASS` / `FAIL` and records the result in `CHECK_LOG`;
* a hard failure raises immediately (so you stop at the mistake, not forty cells later);
* `soft=True` marks a check *advisory* — used twice: for the Krylov GEVP in §8, which is
  genuinely stochastic run to run, and for the noise-damping sanity check in §9. Neither is
  allowed to fail the notebook;
* checks marked `budget_sensitive=True` — the ones whose *tolerance* was calibrated at the
  official shot budget — are downgraded to advisory when `BUDGET != "full"`. Deterministic
  checks (conventions, bit ordering, operator identities) stay hard at every budget.

`sigma_gate(estimate, reference, sem)` is the acceptance test used by Checkpoints 5 and 6:
$|\text{est} - \text{ref}| < 5\,\sigma$.

`show(fig, name)` displays a figure *and* writes it to `./figures/`, so the plots §10.1 asks for as
deliverables exist on disk without you doing anything.

`sub_seed(tag)` turns the single master `SEED` into a reproducible, well-separated seed for each
independent piece of randomness. Note it uses `zlib.crc32`, **not** Python's `hash()` — `hash()` of
a string is randomised per process, which would silently destroy reproducibility between runs. If
you add your own randomised study, route it through `sub_seed` and judging can re-run you exactly.

In [ ]:
# ------------------------------------------------------------------- house style + checks
COL = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "violet": "#4a3aa7",
       "ink": "#0b0b0b", "muted": "#898781", "grid": "#e1e0d9", "axis": "#c3c2b7"}
BG = "#fcfcfb"

def style_axes(ax):
    """One consistent look for every figure in the notebook."""
    ax.set_facecolor(BG)
    ax.grid(True, color=COL["grid"], linewidth=0.8)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(COL["axis"])
    ax.tick_params(colors=COL["muted"], labelsize=9)
    ax.xaxis.label.set_color(COL["ink"])
    ax.yaxis.label.set_color(COL["ink"])
    ax.title.set_color(COL["ink"])

plt.rcParams.update({"figure.facecolor": BG, "savefig.facecolor": BG, "figure.dpi": 110})

FIGDIR = "figures"          # every figure is written here as well as displayed
os.makedirs(FIGDIR, exist_ok=True)

def show(fig, name: str):
    """Display a figure AND save it -- §10.1 asks for these plots as deliverables."""
    fig.savefig(f"{FIGDIR}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()


CHECK_LOG: list[tuple[str, bool, str, bool]] = []

def check(name: str, ok: bool, detail: str = "", soft: bool = False,
          budget_sensitive: bool = False) -> bool:
    """Record and report a checkpoint.

    soft=True             ADVISORY always: reported, never raises. Used for quantities that are
                          genuinely stochastic run to run (the Krylov GEVP is the honest example).
    budget_sensitive=True ADVISORY only when BUDGET != "full", because the tolerance was
                          calibrated at the official shot budget.

    Deterministic checks -- conventions, bit ordering, operator identities -- are neither, so
    they stay HARD at every budget. A green notebook at BUDGET="fast" still proves your circuit
    and your endianness are right; it does not prove your statistics.
    """
    ok = bool(ok)
    soft = soft or (budget_sensitive and BUDGET != "full")
    CHECK_LOG.append((name, ok, detail, soft))
    tag = "PASS" if ok else ("warn" if soft else "FAIL")
    print(f"{tag:>4}  |  {name}" + (f"   ({detail})" if detail else ""))
    if not ok and not soft and STRICT_CHECKS:
        raise AssertionError(f"checkpoint failed: {name} {detail}")
    return ok


def sub_seed(tag: str, index: int = 0) -> int:
    """A reproducible, well-separated seed for `tag`, derived from the master SEED.

    Note we use zlib.crc32, NOT Python's hash(): hash() of a str is randomised per process
    (PYTHONHASHSEED), which would silently destroy reproducibility between runs.
    """
    import zlib
    entropy = [SEED, zlib.crc32(tag.encode()) % (2 ** 31), index]
    return int(np.random.SeedSequence(entropy).generate_state(1)[0]) % (2 ** 31)

def sigma_gate(estimate: float, reference: float, sem: float, n_sigma: float = 5.0):
    """Standard acceptance test used throughout: |est - ref| < n_sigma * sem."""
    dev = abs(estimate - reference)
    z = dev / sem if sem > 0 else np.inf
    return z < n_sigma, f"{estimate:+.4f} vs {reference:+.4f}, {z:.1f} sigma"

print("style + checkpoint helpers ready")

---

# 1 · The physics you need

Read this once now and once more when a sign confuses you at 3 a.m.

## 1.1 The Hadamard test, and the sign convention we use

Prepare the system in $\rho$ (here a pure state $|\psi\rangle$), put the ancilla in
$(|0\rangle+|1\rangle)/\sqrt 2$, apply the controlled evolution $U(t)=e^{-iHt}$, then an ancilla
phase gate $P(\phi)$ with $P(\phi)|1\rangle = e^{i\phi}|1\rangle$, then a final Hadamard, then
measure the ancilla in $Z$:

$$\boxed{\;\langle Z_{\rm anc}\rangle_\phi \;=\; \operatorname{Re}\!\big[e^{i\phi}\operatorname{Tr}(U(t)\rho)\big]\;}$$

so the two phase settings we use throughout give the two quadratures of the same complex signal:

$$\phi = 0 \;\Rightarrow\; \operatorname{Re}\chi(t), \qquad
\phi = -\tfrac\pi2 \;\Rightarrow\; \operatorname{Im}\chi(t), \qquad
\chi(t) \;=\; \operatorname{Tr}\!\big[e^{-iHt}\rho\big].$$

> ### ⚠️ Convention warning — read before you debug
> Reference [1] is internally inconsistent about the sign of $\phi$. Its Eqs. (3), (B2) and
> (D10)–(D12) give $\operatorname{Re}[e^{+i\phi}\operatorname{Tr}(U\rho)]$ — the boxed formula
> above. In the **arXiv version**, whose numbering this notebook uses throughout, its
> Appendix A, Eqs. (A7)–(A9), and the Fig. 1 caption instead say
> $\operatorname{Re}[e^{-i\phi}\operatorname{Tr}(U\rho)]$, i.e. "$\phi = +\pi/2$ gives
> $\operatorname{Im}$". **The boxed convention is the correct one** for
> $P(\phi)|1\rangle=e^{i\phi}|1\rangle$; §3.3 below pins it with a deterministic statevector test.
> If you debug against the appendix sign you will silently reconstruct a **mirror image** of the
> spectrum and lose hours. (This is a known typo in the published paper, not a trap we set.)

## 1.2 What the discarded register actually knows

Write $\rho_{\rm out}$ for the $(1+n)$-qubit state just before any measurement. Reference [1]
tabulates what you get by pairing a single-qubit ancilla Pauli $P_{\rm aux}$ with any system
observable $O$ — their Eq. (C1):

$$\operatorname{Tr}\!\big[(P_{\rm aux}\otimes O_{\rm sys})\,\rho_{\rm out}\big] \;=\; \operatorname{Tr}\!\big[O\,\rho^{(P)}\big],
\qquad \rho^{(P)} \equiv \operatorname{tr}_{\rm aux}\!\big(P_{\rm aux}\rho_{\rm out}\big).$$

Two of those "post-measurement" operators (they are *not* normalised density matrices — they are
outcome-weighted averages you build in classical post-processing) drive this entire challenge:

$$\rho^{(I)}(t) \;=\; \operatorname{tr}_{\rm aux}\rho_{\rm out} \;=\; \tfrac12\big(\rho + U\rho U^\dagger\big) \tag{D1}$$

$$\rho^{(Z)}(t) \;=\; \operatorname{tr}_{\rm aux}\!\big(Z_{\rm aux}\rho_{\rm out}\big) \;=\; \tfrac12\big(e^{i\phi}U\rho + e^{-i\phi}\rho U^\dagger\big) \tag{D10}$$

giving the two identities we will actually estimate:

$$\underbrace{\langle I_{\rm anc}\otimes O\rangle = \operatorname{Tr}\big[O\rho^{(I)}\big]}_{\text{ignore the ancilla}}
\qquad\qquad
\underbrace{\langle Z_{\rm anc}\otimes O\rangle = \operatorname{Re}\big[e^{i\phi}\operatorname{Tr}(O\,U\rho)\big]}_{\text{weight each shot by }a=\pm1} \tag{D2, D11}$$

**Two free lunches fall out of (D1).**

1. $\rho^{(I)}$ does not depend on $\phi$ — so *both* quadratures' records contribute to it.
2. For any $Q$ that commutes with the **implemented** evolution we have $U^\dagger Q U = Q$, hence
   $\operatorname{Tr}[Q\rho^{(I)}(t)] = \operatorname{Tr}[Q\rho]$ **for every $t$** — so such
   observables can be pooled over *all* $N_t\times 2$ settings. At the official budget that is
   $128\times$ the shots (hence $\sqrt{128}\approx 11\times$ tighter error bars) for zero extra
   circuits.

> **Precondition, easy to lose.** "Commutes with the implemented evolution" is not the same as
> "commutes with $H$". With the exact $U=e^{-iHt}$ the set includes $H$ itself. With a **Trotterised**
> $U$ it still includes $Q$ (the $XX$ and $YY$ hopping terms carry equal coefficients and are
> emitted adjacently by the product formula, and $XX+YY$ conserves magnetisation, as do $ZZ$
> and $Z$ — so the whole Trotter step commutes with $Q$ even though the individual $XX$ and
> $YY$ factors do not)
> but **not** $H$: at `reps=1` on this benchmark, pooling $\langle H\rangle$ over all times carries a
> systematic bias of about $-0.02$, roughly $2.6\sigma$ at the official budget. Pool $\langle Q\rangle$
> freely; pool $\langle H\rangle$ only on the exact path, or accept a Trotter bias you must quote.

## 1.3 Local Pauli classical shadows

Each shot, measure every system qubit $j$ in a basis $b_j$ drawn uniformly from $\{X,Y,Z\}$ and
record the triple (basis string $b$, system outcomes $s\in\{\pm1\}^n$, ancilla outcome
$a=\pm1$). For a Pauli string $P$ with support $\operatorname{supp}(P)$ and weight $w(P)$, the
single-shot estimator is

$$\hat P \;=\; 3^{\,w(P)}\prod_{j\in\operatorname{supp}(P)} s_j\,\mathbf 1\!\left[b_j = P_j\right],$$

and the two averages you can take over the *same* records are

$$\mathbb E\big[\hat P\big] = \operatorname{Tr}\!\big[P\rho^{(I)}(t)\big],
\qquad
\mathbb E\big[a\,\hat P\big] = \big\langle Z_{\rm anc}\otimes P\big\rangle_\phi
= \operatorname{Re}\!\big[e^{i\phi}\operatorname{Tr}(P\,U\rho)\big].$$

Any $O=\sum_P c_P P$ follows by linearity, in post-processing, with **no new circuits**. The
single-shot second moment is exactly $3^{w(P)}$, so a weight-2 term costs about $3\times$ the
shots of a weight-1 term for equal accuracy — expect $\langle H\rangle$ (which has $XX$, $YY$,
$ZZ$ terms) to converge more slowly than $\langle Q\rangle$ (all weight 1).

## 1.4 The flagship application: symmetry-resolved spectroscopy

Let $[H,Q]=0$ with common eigenstates $|E_k,q_k\rangle$ and populations
$p_k = |\langle E_k,q_k|\psi\rangle|^2$. Then

$$\chi(t) \;=\; \sum_k p_k\,e^{-iE_kt}, \qquad\qquad
\chi_Q(t) \;=\; \operatorname{Tr}\big[Q\,U(t)\rho\big] \;=\; \sum_k p_k\,q_k\,e^{-iE_kt}.$$

**Same frequencies, different amplitudes.** Reconstruct both series on the same frequency set and
the ratio of amplitudes at peak $k$ *is* the conserved quantum number:

$$\hat q_k \;=\; \frac{A^Q_k}{A_k} \;\approx\; q_k .$$

> **The ancilla tells you *which energies*. The ancilla-correlated garbage register tells you
> *which symmetry sector*.** One set of experiments, two spectroscopies.

**Degeneracy caveat.** If two states with different $q$ share one unresolved energy, the ratio
returns their population-weighted average, not separate labels. In the benchmark below the
*populated* levels are non-degenerate and well separated; the full spectrum does contain
unpopulated cross-sector levels near $E\approx 0.45$, harmless here because $|\psi\rangle$ has no
support on them.

---

# 2 · The benchmark model

A three-qubit open-boundary XXZ chain with local fields:

$$H \;=\; 0.65\sum_{i=0}^{1}\big(X_iX_{i+1}+Y_iY_{i+1}\big) \;+\; 0.25\sum_{i=0}^{1}Z_iZ_{i+1}
\;+\; 0.40\,Z_0 \;-\; 0.50\,Z_1 \;+\; 0.15\,Z_2 ,$$

with conserved total magnetisation $Q = Z_0+Z_1+Z_2$ (the $XX+YY$ hopping preserves the number of
up-spins, so $[H,Q]=0$), and input state $|\psi\rangle = R_y^{(0)}(1.3)\,|000\rangle$.

That state is a superposition of $|000\rangle$ — which lives alone in the $Q=+3$ sector and is
therefore an exact eigenstate with $E=0.55$ — and $|001\rangle$ (endianness rule #1: the flipped
qubit is qubit 0, so it is the *rightmost* character), which spreads over the three levels of the
$Q=+1$ sector. So the spectrum you must recover has **one strong line in one sector
and three weaker lines in another**: exactly the situation where symmetry labels are useful.

In [ ]:
N_SYS = 3                       # system qubits; the ancilla is one extra qubit (index N_SYS)
PHI_RE, PHI_IM = 0.0, -np.pi / 2   # the two quadrature settings

PAULI_CODES = {"X": 0, "Y": 1, "Z": 2}
CODE_TO_PAULI = "XYZ"


def benchmark_hamiltonian() -> SparsePauliOp:
    """H = 0.65 sum (XX + YY) + 0.25 sum ZZ + 0.40 Z0 - 0.50 Z1 + 0.15 Z2 (open chain)."""
    terms = []
    for i in range(N_SYS - 1):
        terms += [("XX", [i, i + 1], 0.65), ("YY", [i, i + 1], 0.65), ("ZZ", [i, i + 1], 0.25)]
    terms += [("Z", [0], 0.40), ("Z", [1], -0.50), ("Z", [2], 0.15)]
    return SparsePauliOp.from_sparse_list(terms, num_qubits=N_SYS).simplify()


def conserved_charge() -> SparsePauliOp:
    """Q = Z_0 + Z_1 + Z_2, the total magnetisation.  [H, Q] = 0."""
    return SparsePauliOp.from_sparse_list(
        [("Z", [j], 1.0) for j in range(N_SYS)], num_qubits=N_SYS)


def state_prep_circuit() -> QuantumCircuit:
    """|psi> = Ry(1.3) on qubit 0, applied to |000>."""
    qc = QuantumCircuit(N_SYS, name="prep")
    qc.ry(1.3, 0)
    return qc


HAM = benchmark_hamiltonian()
CHARGE = conserved_charge()
Z0_OBS = SparsePauliOp.from_sparse_list([("Z", [0], 1.0)], num_qubits=N_SYS)   # NOT conserved

print("H =", HAM)
print()
print("Q =", CHARGE)

> **Endianness rule #1.** Build Pauli operators with
> `SparsePauliOp.from_sparse_list([("XX", [0, 1], 0.65), ...], num_qubits=3)` and **never** by
> typing label strings by hand. In a Qiskit label like `"IZX"` the *rightmost* character is
> qubit 0. Getting this wrong flips which qubit carries the $+0.40$ field and quietly changes
> your spectrum.

## 2.1 Exact references (validation only — never inside an estimator)

Because $Q$ is diagonal in the computational basis, we can build a common $(H,Q)$ eigenbasis by
diagonalising $H$ *block by block* inside each charge sector. That is more robust than
diagonalising $H$ globally and then trying to read off $Q$, which fails whenever $H$ has
degeneracies across sectors.

In [ ]:
def exact_state(prep: QuantumCircuit | None = None) -> np.ndarray:
    return np.asarray(Statevector(prep if prep is not None else state_prep_circuit()).data)


@dataclass
class ExactSpectrum:
    """Symmetry-resolved exact eigensystem of (H, Q) plus the populations of |psi>."""
    energies: np.ndarray      # (2**n,)
    charges: np.ndarray       # (2**n,) integer Q labels
    populations: np.ndarray   # (2**n,) p_k = |<E_k, q_k | psi>|^2

    def populated(self, threshold: float = 1e-10):
        m = self.populations > threshold
        return self.energies[m], self.charges[m], self.populations[m]


def exact_spectrum(ham=None, charge=None, psi=None) -> ExactSpectrum:
    """Block-diagonalise H inside each Q sector -> a common (H, Q) eigenbasis."""
    ham = ham if ham is not None else HAM
    charge = charge if charge is not None else CHARGE
    psi = psi if psi is not None else exact_state()
    hmat = ham.to_matrix()
    qdiag = np.real(np.diag(charge.to_matrix()))
    energies, charges, pops = [], [], []
    for q in sorted(set(np.round(qdiag).astype(int))):
        idx = np.where(np.round(qdiag).astype(int) == q)[0]
        evals, evecs = np.linalg.eigh(hmat[np.ix_(idx, idx)])
        amps = evecs.conj().T @ psi[idx]
        energies += list(evals)
        charges += [q] * len(idx)
        pops += list(np.abs(amps) ** 2)
    order = np.argsort(energies)
    return ExactSpectrum(np.array(energies)[order],
                         np.array(charges)[order],
                         np.array(pops)[order])


def exact_unitary(ham: SparsePauliOp, t: float) -> np.ndarray:
    """U(t) = exp(-i H t) by eigendecomposition (exact for any t)."""
    evals, evecs = np.linalg.eigh(ham.to_matrix())
    return (evecs * np.exp(-1j * evals * t)) @ evecs.conj().T


def exact_chi(ham, psi, ts) -> np.ndarray:
    """chi(t) = <psi| e^{-iHt} |psi> = sum_k p_k e^{-i E_k t}."""
    evals, evecs = np.linalg.eigh(ham.to_matrix())
    p = np.abs(evecs.conj().T @ psi) ** 2
    return np.exp(-1j * np.outer(np.asarray(ts), evals)) @ p


def exact_chi_O(ham, psi, obs: SparsePauliOp, ts) -> np.ndarray:
    """chi_O(t) = Tr[O U(t) |psi><psi|] = <psi| O U(t) |psi>."""
    evals, evecs = np.linalg.eigh(ham.to_matrix())
    phi_o = obs.to_matrix().conj().T @ psi          # O is Hermitian
    weights = np.conj(evecs.conj().T @ phi_o) * (evecs.conj().T @ psi)
    return np.exp(-1j * np.outer(np.asarray(ts), evals)) @ weights


def exact_system_marginal_expectation(ham, psi, obs: SparsePauliOp, t: float) -> float:
    """Tr[O rho^(I)(t)] with rho^(I) = (rho + U rho U^dag)/2  -- Eq. (D1)."""
    psi_t = exact_unitary(ham, t) @ psi
    omat = obs.to_matrix()
    return float(0.5 * np.real(psi.conj() @ omat @ psi + psi_t.conj() @ omat @ psi_t))


PSI = exact_state()
SPEC = exact_spectrum()
E_EXACT, Q_EXACT_LABELS, P_EXACT = SPEC.populated(1e-6)

H_EXPECT_EXACT = float(np.real(PSI.conj() @ HAM.to_matrix() @ PSI))
Q_EXPECT_EXACT = float(np.real(PSI.conj() @ CHARGE.to_matrix() @ PSI))

print("Populated levels of |psi>  (these are the four lines you must recover):\n")
print("      E_k        q_k       p_k")
for e, q, p in zip(E_EXACT, Q_EXACT_LABELS, P_EXACT):
    print(f"   {e:+9.4f}   {q:+4d}    {p:.4f}")
print(f"\n<H> = {H_EXPECT_EXACT:+.6f}      <Q> = {Q_EXPECT_EXACT:+.6f}")
print(f"full spectral radius   = {np.max(np.abs(SPEC.energies)):.3f}")
print(f"populated-line radius  = {np.max(np.abs(E_EXACT)):.3f}")
print(f"min populated spacing  = {np.min(np.diff(np.sort(E_EXACT))):.3f}")

### ✅ Checkpoint 2 — the benchmark is what we say it is

Before trusting a single circuit, verify the *classical* setup. These eight asserts confirm that
$[H,Q]=0$ to machine precision, that the state populates exactly four levels across two charge
sectors, that $|000\rangle$ really is an exact eigenstate at $E=0.55$ with weight $\cos^2(0.65)$,
and — the two that people forget — that the chosen time grid neither aliases the spectrum nor
under-resolves the closest pair of populated lines.

If you later change the model or the grid, this cell tells you immediately whether the challenge
is still well posed.

In [ ]:
# ---------------------------------------------------- CHECKPOINT 2: the model is what we claim
h_mat, q_mat = HAM.to_matrix(), CHARGE.to_matrix()

check("[H, Q] = 0 exactly",
      np.linalg.norm(h_mat @ q_mat - q_mat @ h_mat) < 1e-12,
      f"||[H,Q]|| = {np.linalg.norm(h_mat @ q_mat - q_mat @ h_mat):.2e}")

check("populations sum to 1",
      np.isclose(SPEC.populations.sum(), 1.0, atol=1e-10))

check("exactly four populated levels", len(E_EXACT) == 4, f"got {len(E_EXACT)}")

check("populated sectors are {+1, +3}",
      set(np.round(Q_EXACT_LABELS).astype(int)) == {1, 3})

check("|000> is an exact eigenstate at E = 0.55",
      np.isclose(E_EXACT[np.round(Q_EXACT_LABELS).astype(int) == 3][0], 0.55, atol=1e-12))

check("Q=3 weight equals cos^2(0.65)",
      np.isclose(P_EXACT[np.round(Q_EXACT_LABELS).astype(int) == 3].sum(),
                 np.cos(0.65) ** 2, atol=1e-10),
      f"{P_EXACT[np.round(Q_EXACT_LABELS).astype(int) == 3].sum():.6f}")

check("no aliasing at this dt",
      np.max(np.abs(SPEC.energies)) < np.pi / DT,
      f"radius {np.max(np.abs(SPEC.energies)):.2f} < Nyquist {np.pi / DT:.2f}")

check("populated lines are Fourier-resolvable",
      np.min(np.diff(np.sort(E_EXACT))) > 2 * np.pi / T_MAX,
      f"spacing {np.min(np.diff(np.sort(E_EXACT))):.2f} > resolution {2 * np.pi / T_MAX:.2f}")

---

# 3 · Building the circuit in Qiskit

The circuit is short but every piece has a trap attached:

```
 |0>_anc ──[H]──●──[P(phi)]──[H]────────────────[M]  -> a = +-1
                │
 |0>_q0 ─[Ry]───┼──────────────────[X/Y/Z basis]─[M]  ┐
 |0>_q1 ────────┤ e^{-iHt}  ───────[X/Y/Z basis]─[M]  ├ shadow snapshot s, b
 |0>_q2 ────────┴──────────────────[X/Y/Z basis]─[M]  ┘
```

*(The schematic puts the ancilla on top for readability. In the Qiskit drawing further down it is
the **bottom** wire, because `QuantumCircuit(sys_reg, anc_reg)` declares the system register
first. Same circuit, different wire order.)*

## 3.1 Controlled time evolution — and the ordering trap

Qiskit's own `PauliEvolutionGate` documentation warns that *"the order in which the approximation
and methods like `control()` and `power()` are called matters."* There are two genuinely different
objects you can build:

| what you build | what it is |
|---|---|
| `PauliEvolutionGate(H, t).control(1)` after synthesis | **controlled(approximation)** — an *exactly* controlled gate whose target is the Trotterised $U_{\rm trot}(t)$ |
| controlling each Trotter step's generator | **approximation(controlled)** — a product formula for the controlled generator; a *different* unitary |

We build the first one: synthesise the product formula, *then* control the resulting circuit. That
way the ancilla branch structure $|0\rangle\langle0|\otimes I + |1\rangle\langle1|\otimes
U_{\rm trot}$ is exact, and the only error is the Trotter error in $U_{\rm trot}$ itself — which is
what all the theory in §1 assumes.

`method="exact"` instead builds that block matrix numerically. It is simulator-only (not
hardware-native) but has **zero** Trotter error, so we use it for all validation.

> **Do not read a hardware conclusion off the resource table in §6.4.** At $n=3$ the transpiler
> synthesises the exact 4-qubit `UnitaryGate` into *fewer* CX gates than the Trotter circuit needs —
> the "simulator-only" path looks cheaper. That is an artefact of the tiny system: exact synthesis
> costs $O(4^n)$ two-qubit gates, while a product formula costs $O(\text{poly}(n))$ per step. The
> Trotter advantage is asymptotic; the crossover sits well below the sizes anyone runs on
> hardware, but it is not at $n=3$.

> ## 🎯 Challenge 1 — controlled time evolution
>
> **Build** `build_controlled_evolution(ham, t, method, reps) -> Gate`, a controlled-$U(t)$ whose
> **first** qubit is the ancilla control, so it can be appended as
> `qc.append(gate, [ancilla, sys_0, ..., sys_{n-1}])`.
>
> **Two paths, both required.**
> `method="exact"` returns the numerically exact block unitary
> $|0\rangle\langle0|\otimes I + |1\rangle\langle1|\otimes e^{-iHt}$ as a `UnitaryGate`. Mind the
> ordering: with qargs `[anc, s0, …]` the ancilla is the *lowest* matrix bit, so the kron factors
> go `np.kron(np.eye(2**n), p0) + np.kron(u, p1)`.
> `method="trotter"` synthesises a 2nd-order Suzuki–Trotter product formula **first**, transpiles it
> to a concrete gate set, and only **then** calls `.control(1)`.
>
> ⚠️ **Trap.** The dangerous move is *approximation-of-controlled*: Trotterising the controlled
> generator $|1\rangle\langle1|\otimes H$ instead of controlling the synthesised $U_{\rm trot}$.
> That is a different unitary and the theory in §1 no longer applies to it. Qiskit's docstring
> warning — *"the order in which the approximation and methods like `control()` and `power()` are
> called matters"* — is about exactly that. Note that `PauliEvolutionGate(...).control(1)` is
> **not** the trap: Qiskit defers synthesis, so it yields the same object we build here.
>
> **You will know it works when** Checkpoint 3a is green: the exact path matches the block matrix to
> $10^{-10}$, and the Trotter error decreases monotonically with `reps`.

In [ ]:
def build_controlled_evolution(ham: SparsePauliOp, t: float,
                               method: str = "exact", reps: int = 2) -> Gate:
    """Controlled-U(t) whose FIRST qubit is the ancilla control.

    method="exact"   : the block unitary |0><0| (x) I + |1><1| (x) e^{-iHt}, built numerically.
                       Simulator-only, zero Trotter error -- use this for validation.
    method="trotter" : synthesise the 2nd-order Suzuki-Trotter product formula FIRST, then
                       control the whole synthesised circuit.  Hardware-realistic.

    Append with:  qc.append(gate, [ancilla, sys_0, ..., sys_{n-1}]).
    """
    # ================================================================== TODO
    # Challenge 1: implement build_controlled_evolution().
    # if method == "exact":  build |0><0| (x) I + |1><1| (x) exp(-iHt) as a UnitaryGate.
    #     With qargs [anc, s0, ..., s_{n-1}] the ancilla is the LOWEST matrix bit, so
    #     cu = np.kron(np.eye(2**n), p0) + np.kron(u, p1)  with p0, p1 = |0><0|, |1><1|.
    # if method == "trotter": PauliEvolutionGate(ham, time=t, synthesis=SuzukiTrotter(order=2,
    #     reps=reps)) -> put it in a QuantumCircuit -> transpile to a concrete basis ->
    #     .to_gate().control(1).  Synthesise FIRST, control SECOND.
    # ==================================================================
    raise NotImplementedError("Challenge 1: implement build_controlled_evolution()")

## 3.2 The shadow-Hadamard circuit

`basis` is a length-$n$ list with entries `0 = X`, `1 = Y`, `2 = Z` selecting each system qubit's
measurement basis; the corresponding pre-measurement rotations are $H$ for $X$ and
$HS^\dagger$ for $Y$ (that is, `sdg` then `h` in circuit order). Passing `basis=None,
measure=False` gives the bare circuit for exact
statevector checks.

> ### ⚠️ Trap — endianness rule #2 (a *different* convention from rule #1)
>
> We put the system on qubits $0..n-1$ and the
> ancilla on qubit $n$, with one classical register `c[0..n]`. In a memory bitstring the
> **leftmost** character is the **highest** classical bit — so the ancilla is character `0` and
> system qubit $j$ is character `n - j`. Two different orderings in one project; §4.2 has a unit
> test for this one.

> ## 🎯 Challenge 2 — the shadow-Hadamard circuit *(given complete — read it, do not rewrite it)*
>
> **Read** `build_shadow_hadamard_circuit(ham, t, phi, basis, ...) -> QuantumCircuit`: state prep,
> ancilla Hadamard, controlled evolution, phase gate $P(\phi)$, second ancilla Hadamard, then the
> per-qubit shadow basis rotation, then measurement of all $n+1$ qubits.
>
> **Spec.**
> `basis` is a length-$n$ sequence of `0/1/2` = $X/Y/Z$; apply $H$ for $X$, $HS^\dagger$ for $Y$
> (`sdg` then `h` in circuit order),
> nothing for $Z$. `basis=None, measure=False` must give the bare circuit for statevector checks.
> Wire the classical register so that `c[j]` holds system qubit $j$ and `c[n]` holds the ancilla.
>
> ⚠️ **Trap.** This is the second of the notebook's two endianness conventions, and it is *not* the
> same as the Pauli-label one. Read the *Endianness rule #2* callout immediately above before
> you debug anything.
>
> **You will know it works when** Checkpoint 3b (§3.3) is green. That cell verifies
> $\langle Z_{\rm anc}\otimes O\rangle_\phi = \operatorname{Re}[e^{i\phi}\operatorname{Tr}(O\,U\rho)]$
> deterministically for five observables, three times and both quadratures. One green cell pins your
> $\phi$ sign, your evolution direction, your qubit ordering and your $U = e^{-iHt}$ convention
> simultaneously.

In [ ]:
def build_shadow_hadamard_circuit(ham: SparsePauliOp, t: float, phi: float,
                                  basis: Sequence[int] | None,
                                  prep: QuantumCircuit | None = None,
                                  method: str = "exact", reps: int = 2,
                                  measure: bool = True) -> QuantumCircuit:
    """Hadamard test + randomised local-Pauli measurement of the system register."""
    n = ham.num_qubits
    prep = prep if prep is not None else state_prep_circuit()
    sys_reg = QuantumRegister(n, "sys")
    anc_reg = QuantumRegister(1, "anc")
    qc = QuantumCircuit(sys_reg, anc_reg)

    qc.compose(prep, qubits=sys_reg, inplace=True)          # |psi>
    qc.h(anc_reg[0])                                        # ancilla -> |+>
    qc.append(build_controlled_evolution(ham, t, method, reps), [anc_reg[0], *sys_reg])
    if phi != 0.0:
        qc.p(phi, anc_reg[0])                               # quadrature selector
    qc.h(anc_reg[0])                                        # interfere

    if basis is not None:                                   # shadow basis rotation
        for j, b in enumerate(basis):
            if b == 0:            # X basis
                qc.h(sys_reg[j])
            elif b == 1:          # Y basis
                qc.sdg(sys_reg[j])
                qc.h(sys_reg[j])
            # b == 2 (Z basis): nothing to do

    if measure:
        creg = ClassicalRegister(n + 1, "c")
        qc.add_register(creg)
        for j in range(n):
            qc.measure(sys_reg[j], creg[j])                 # c[j] <- system qubit j
        qc.measure(anc_reg[0], creg[n])                     # c[n] <- ancilla
    return qc


demo = build_shadow_hadamard_circuit(HAM, 0.9, PHI_IM, basis=[0, 1, 2], method="trotter", reps=1)
_demo_t = transpile(demo, basis_gates=["rz", "sx", "x", "cx"], optimization_level=1,
                    seed_transpiler=0)
print(f"Trotter reps=1 -- as built : depth {demo.depth():3d}   {dict(demo.count_ops())}")
print(f"               -- transpiled: depth {_demo_t.depth():3d}   "
      f"{_demo_t.count_ops().get('cx', 0)} CX   (this is the number that matters)")

# §10.1 asks for circuit diagrams of BOTH quadratures -- save them while we are here
for _name, _phi in (("phi_0_real", PHI_RE), ("phi_-pi2_imag", PHI_IM)):
    _f = build_shadow_hadamard_circuit(HAM, 0.9, _phi, basis=[0, 1, 2],
                                       method="trotter", reps=1).draw("mpl", fold=-1,
                                                                      style="clifford")
    _f.savefig(f"{FIGDIR}/03_circuit_{_name}.png", dpi=200, bbox_inches="tight")
    plt.close(_f)

demo.draw("mpl", fold=-1, style="clifford")

### ✅ Checkpoint 3a — the controlled evolution is genuinely controlled

Two things to prove. First, the `"exact"` path must equal the block matrix
$|0\rangle\langle0|\otimes I + |1\rangle\langle1|\otimes e^{-iHt}$ *as an operator* — not
approximately, exactly. Second, the `"trotter"` path must be an **exactly controlled approximation**:
its error against the true block matrix must shrink with `reps`, which is only true if you
controlled the synthesised circuit rather than synthesising a controlled generator.

A caveat on what this checkpoint can and cannot see. `PauliEvolutionGate(H, t, synthesis=...)`
defers synthesis, so calling `.control(1)` on it gives *exactly* the object we build here — same
unitary to $10^{-12}$, same error sequence. That is **not** the trap. The genuinely different
object is Trotterising the *controlled generator* $|1\rangle\langle1|\otimes H$, whose errors run
$0.39 / 0.073 / 0.0047$ at `reps = 1 / 2 / 8` against this route's $0.82 / 0.168 / 0.0097$: a
different unitary, converging to the same limit by a different path. Checkpoint 3a would not fail
on it — only reading your own circuit tells you which one you built.

In [ ]:
# ------------------------------------------------- CHECKPOINT 3a: the circuit is the right one
def _embed_controlled(u: np.ndarray) -> np.ndarray:
    """|0><0|_anc (x) I + |1><1|_anc (x) U with the ancilla as qubit 3 (highest)."""
    p0, p1 = np.diag([1.0, 0.0]), np.diag([0.0, 1.0])
    return np.kron(p0, np.eye(u.shape[0])) + np.kron(p1, u)


t_test = 0.73
qc_ex = QuantumCircuit(4)
qc_ex.append(build_controlled_evolution(HAM, t_test, "exact"), [3, 0, 1, 2])
_blk = _embed_controlled(exact_unitary(HAM, t_test))
check("c-U(exact) equals the block matrix",
      np.allclose(Operator(qc_ex).data, _blk, atol=1e-10),
      f"max |c-U - block| = {np.abs(Operator(qc_ex).data - _blk).max():.2e}  "
      f"(swapped kron factors put the ancilla in the wrong slot; a sign error in exp(-iHt) "
      f"leaves the diagonal block right and the controlled block conjugated)")

# Trotter path: exactly controlled, and converging in reps
errs = []
for reps in (1, 2, 8):
    qc_tr = QuantumCircuit(4)
    qc_tr.append(build_controlled_evolution(HAM, 1.1, "trotter", reps=reps), [3, 0, 1, 2])
    errs.append(np.linalg.norm(Operator(qc_tr).data - _embed_controlled(exact_unitary(HAM, 1.1)), 2))
check("Trotter error decreases with reps", errs[2] < errs[1] < errs[0],
      "  ".join(f"reps={r}: {e:.2e}" for r, e in zip((1, 2, 8), errs)))
check("2nd-order Trotter at reps=8 is accurate", errs[2] < 2e-2, f"{errs[2]:.2e}")

## 3.3 ✅ Checkpoint 3b — pinning every sign convention, exactly

This is the single most important cell in the notebook, and it is worth understanding *why* rather
than just running it. It verifies

$$\big\langle Z_{\rm anc}\otimes O\big\rangle_\phi \;=\; \operatorname{Re}\!\big[e^{i\phi}\operatorname{Tr}(O\,U(t)\rho)\big]$$

for five observables (including the identity, a single-qubit $Z$, the conserved charge, the
Hamiltonian and a two-qubit $XX$), at three times, in both quadratures — thirty independent
identities in one cell.

**It uses no shots.** Both sides are computed from the exact statevector of the circuit, so the
comparison is at machine precision ($10^{-10}$ tolerance, actual deviation $\sim10^{-15}$) and the
result is deterministic. That matters: a shot-based test can only tell you a sign is wrong once the
statistical error is smaller than the discrepancy, and for the *imaginary* quadrature a flipped
sign at some times looks exactly like noise. A statevector test catches it instantly and always.

**What one green cell buys you.** Four independent conventions have to be simultaneously right for
this identity to hold — the sign in $P(\phi)$, the direction of the controlled evolution
($e^{-iHt}$ versus $e^{+iHt}$), which qubit is the ancilla, and the Pauli-label endianness of $O$.
Break any one of them and the identity fails somewhere in the loop. Break *two* and they can
cancel at $t=0$ but not at $t = 2.3$, which is why the test sweeps several times rather than one.

**Then two more identities.** The cell also confirms $\operatorname{tr}_{\rm anc}\rho_{\rm out} =
(\rho + U\rho U^\dagger)/2$ — Eq. (D1), the foundation of every unweighted-shadow estimate — and
that inserting the shadow basis rotations leaves the ancilla marginal completely untouched, which
is the formal statement that measuring the garbage register costs you nothing.

If you change anything about the circuit, re-run this cell before you re-run anything else.

In [ ]:
# ------------------------------------------- CHECKPOINT 3b: every sign convention, exactly
z_anc_1q = SparsePauliOp.from_sparse_list([("Z", [0], 1.0)], num_qubits=1)
test_obs = {
    "I":    SparsePauliOp.from_sparse_list([("I", [0], 1.0)], num_qubits=N_SYS),
    "Z0":   Z0_OBS,
    "Q":    CHARGE,
    "H":    HAM,
    "X0X1": SparsePauliOp.from_sparse_list([("XX", [0, 1], 1.0)], num_qubits=N_SYS),
}

worst = 0.0
for t in (0.0, 0.7, 2.3):
    for phi in (PHI_RE, PHI_IM):
        sv = Statevector(build_shadow_hadamard_circuit(HAM, t, phi, basis=None, measure=False))
        for name, obs in test_obs.items():
            joint = z_anc_1q.tensor(obs)                    # ancilla = highest qubit
            measured = float(np.real(sv.expectation_value(joint)))
            expected = float(np.real(np.exp(1j * phi) * exact_chi_O(HAM, PSI, obs, [t])[0]))
            worst = max(worst, abs(measured - expected))

check("<Z_anc (x) O>_phi = Re[e^{i phi} Tr(O U rho)]  for all (t, phi, O)",
      worst < 1e-10, f"max deviation {worst:.2e}")

# the (D1) marginal: tracing out the ancilla gives (rho + U rho U+)/2, independent of phi
sv = Statevector(build_shadow_hadamard_circuit(HAM, 1.3, PHI_IM, basis=None, measure=False))
rho_sys = partial_trace(sv, [N_SYS]).data
u13 = exact_unitary(HAM, 1.3)
rho0 = np.outer(PSI, PSI.conj())
check("tr_anc(rho_out) = (rho + U rho U+)/2   [Eq. D1]",
      np.allclose(rho_sys, 0.5 * (rho0 + u13 @ rho0 @ u13.conj().T), atol=1e-10),
      f"||diff|| = {np.linalg.norm(rho_sys - 0.5 * (rho0 + u13 @ rho0 @ u13.conj().T)):.2e}")

# the shadow basis rotation must not disturb the ancilla marginal
p_plain = Statevector(
    build_shadow_hadamard_circuit(HAM, 0.9, PHI_RE, basis=None, measure=False)).probabilities([N_SYS])
ok_rot = all(np.allclose(p_plain, Statevector(
    build_shadow_hadamard_circuit(HAM, 0.9, PHI_RE, basis=b, measure=False)
).probabilities([N_SYS]), atol=1e-12) for b in ([0, 1, 2], [1, 1, 1], [0, 0, 0], [2, 2, 2]))
check("the shadow basis rotation leaves the ancilla marginal untouched", ok_rot,
      "the full 'measurement does not disturb' statement is proved in checkpoint 3c")

---

## 3.4 Measurement design: do we need mid-circuit measurement? Does Qiskit have a shadows API?

Two questions come up immediately when you try to implement Reference [1] literally. Both have
clean answers, and both save you a lot of work.

### Question 1 — "the paper talks about *post-measurement* states. Must I measure the ancilla first?"

**No. Order does not matter, and a single terminal measurement of all $n+1$ qubits is the correct
and cheapest implementation.**

Here is why, in one line. Let $\{M_a\}$ be the ancilla POVM (a $Z$ measurement) and $\{N_s\}$ the
system POVM (a rotated computational-basis measurement). They act on **disjoint tensor factors**,
so as operators on the joint space $M_a\otimes I$ and $I\otimes N_s$ commute. The Lüders rule for
"ancilla first, then system" gives

$$p(a,s) \;=\; \operatorname{Tr}\!\big[(I\otimes N_s)\,(M_a\otimes I)\,\rho_{\rm out}\,(M_a\otimes I)\big]
\;\overset{\text{cyclicity}+\,M_a^2=M_a}{=}\; \operatorname{Tr}\!\big[(M_a\otimes N_s)\,\rho_{\rm out}\big],$$

which is exactly the single global Born rule — and, by the same computation with the roles
swapped, exactly what "system first, then ancilla" gives too. This *is* the paper's Eq. (C1),
$\operatorname{Tr}[(P_{\rm aux}\otimes O_{\rm sys})\rho_{\rm out}] = \operatorname{Tr}[O\rho^{(P)}]$:
the left-hand side is a **joint expectation on the unmeasured output state**, i.e. the global
measurement picture.

So the paper's $\rho^{(I)}$ and $\rho^{(Z)}$ are not physical states you have to prepare — they
are **classical re-weightings of one shot table**:

* ignore the ancilla column → you are sampling $\rho^{(I)}$;
* multiply each snapshot by the ancilla outcome $a=\pm1$ → you are sampling $\rho^{(Z)}$.

One experiment, two post-measurement operators, zero extra circuits. (The paper's $\rho^{(X)}$ and
$\rho^{(Y)}$ need the *ancilla* read out in a different basis — for $X$ that just means **omitting
the final Hadamard**, per Appendix B. That is the basis of advanced Track B.)

The cell below proves all of this numerically: operator-algebraically, and then by actually
running a mid-circuit-measurement circuit against a terminal-measurement circuit in Aer.

> ### Where the equivalence stops
> The argument assumes **ideal, projective, QND, crosstalk-free** readout with nothing acting
> between the two measurements. On real hardware none of that is exactly true: mid-circuit readout
> dephases spectator qubits, is not perfectly QND, and measurement crosstalk correlates the two
> registers — so the two orders give *measurably different* distributions. The equivalence also
> breaks outright if the ancilla is reset and reused, or if any conditional operation sits between
> the measurements (Tracks B and E can both get there). Being precise: Eq. (C1) is an operator
> identity about $\rho_{\rm out}$ that holds whether or not you measure at all; a terminal global
> measurement is simply the cheapest unbiased way to *estimate* its left-hand side, and only for
> $P_{\rm aux}\in\{I, Z\}$ — $X$ and $Y$ need the ancilla rotated first.

### Question 2 — "does Qiskit ship classical shadows?"

**No official Qiskit API exists.** As surveyed in August 2026 (re-check before you rely on it):

| where you might look | verdict |
|---|---|
| `qiskit` core 2.5.x | no shadow module; core issue [#8581 "Shadow Tomography"](https://github.com/Qiskit/qiskit/issues/8581) is open and unactioned since 2022 |
| `qiskit-aer` 0.17.x | no |
| `qiskit-experiments` 0.14.x | only `StateTomography` / `ProcessTomography` / RB / readout mitigation — **no** shadows |
| `qiskit-addon-*` (obp, sqd, mpf, aqc-tensor, cutting, …) | none provide shadows |
| `qiskit.primitives` | no shadow estimation. (Twirling *does* exist — `qiskit.circuit.twirling.pauli_twirl_2q_gates`, and `TwirlingOptions` in qiskit-ibm-runtime with `enable_measure` — but that is randomised *compilation*, not shadow tomography) |
| [`povm-toolbox`](https://github.com/qiskit-community/povm-toolbox) (qiskit-**community**, v0.2.0) | **yes** — `ClassicalShadows`, `LocallyBiasedClassicalShadows`, plus `RandomizedProjectiveMeasurements`, `MutuallyUnbiasedBasesMeasurements`, `DilationMeasurements`; no global-Clifford shadows. Community research code, not an IBM-supported addon |
| PennyLane, for contrast | first-class and differentiable: `qml.classical_shadow`, `qml.shadow_expval` |

**What that means for you:** the shadow estimator is ~60 lines of NumPy and we write it from
scratch in §5. That is deliberate — you should understand the $3^{w}$ factor and the basis-match
indicator rather than import them. If you want to cross-check against a library,
`pip install povm-toolbox` and compare; a validated cross-check is a legitimate originality point.

There is also **no deferred-measurement transpiler pass** in Qiskit (no `DeferMeasurements`); the
only related pass is `qiskit_ibm_runtime.transpiler.passes.ConvertToMidCircuitMeasure`, which goes
the other way. Since §3.4 shows we do not need mid-circuit measurement at all, this is moot for
this challenge — but worth knowing if you take Track B or E.

> ### ⚠️ If you *do* use Aer `save_*` instructions with `conditional=True`
> `transpile()` will happily reorder a `save_density_matrix(..., conditional=True)` to *before* a
> non-overlapping mid-circuit `measure`, silently giving you the pre-measurement averaged state.
> Insert `qc.barrier()` between the measure and the save. (Verified at optimisation levels 0–3.)

### ✅ Checkpoint 3c — measurement order really is irrelevant

Three independent proofs of the claim above, in increasing concreteness. First that the ancilla and
system POVMs commute as operators; then that the joint distribution $p(a,s)$ comes out identical
whether you apply the Lüders update ancilla-first, system-first, or not at all; then — in the next
cell — an actual Aer experiment.

In [ ]:
# ------------------- CHECKPOINT 3c: measurement order is irrelevant (three independent proofs)
def _rho_out(t, phi):
    sv = np.asarray(Statevector(
        build_shadow_hadamard_circuit(HAM, t, phi, basis=None, measure=False)).data)
    return np.outer(sv, sv.conj())


def _basis_rotation(basis):
    qc = QuantumCircuit(N_SYS)
    for j, b in enumerate(basis):
        if b == 0:
            qc.h(j)
        elif b == 1:
            qc.sdg(j); qc.h(j)
    return Operator(qc).data


def _povms(basis):
    v = _basis_rotation(basis)
    m = [np.kron(np.diag([1.0, 0.0]), np.eye(2 ** N_SYS)),      # ancilla = highest qubit
         np.kron(np.diag([0.0, 1.0]), np.eye(2 ** N_SYS))]
    n = []
    for s in range(2 ** N_SYS):
        proj = np.zeros((2 ** N_SYS, 2 ** N_SYS)); proj[s, s] = 1.0
        n.append(np.kron(np.eye(2), v.conj().T @ proj @ v))
    return m, n


T_ORD, BASIS_ORD = 0.9, [0, 1, 2]
rho_o = _rho_out(T_ORD, PHI_IM)
M, Nn = _povms(BASIS_ORD)

p_global   = np.array([[np.real(np.trace(M[a] @ Nn[s] @ rho_o)) for s in range(8)] for a in range(2)])
p_anc_1st  = np.array([[np.real(np.trace(Nn[s] @ (M[a] @ rho_o @ M[a]))) for s in range(8)] for a in range(2)])
p_sys_1st  = np.array([[np.real(np.trace(M[a] @ (Nn[s] @ rho_o @ Nn[s]))) for s in range(8)] for a in range(2)])
max_comm   = max(np.linalg.norm(M[a] @ Nn[s] - Nn[s] @ M[a]) for a in range(2) for s in range(8))

check("ancilla and system POVMs commute", max_comm < 1e-12, f"max ||[M_a, N_s]|| = {max_comm:.1e}")
check("p(a,s): global == ancilla-measured-first",
      np.allclose(p_global, p_anc_1st, atol=1e-12), f"max diff {np.abs(p_global - p_anc_1st).max():.1e}")
check("p(a,s): global == system-measured-first",
      np.allclose(p_global, p_sys_1st, atol=1e-12), f"max diff {np.abs(p_global - p_sys_1st).max():.1e}")

The algebra above is airtight, but algebra is not an experiment. This cell builds a **genuinely
different circuit** — the ancilla measured mid-circuit, then barriers, then the basis rotation, then
the system measurement — and runs both versions on Aer at 200 000 shots. If order mattered, the two
outcome distributions would differ; they do not, and both sit within the shot-noise floor of the
exact distribution computed above.

Note the two `qc.barrier()` calls. Without them the transpiler is free to slide the basis rotations
back across the mid-circuit measurement, which would make the comparison meaningless (and is
exactly the hazard described in the callout above).

In [ ]:
# ...and the same statement as an actual Aer experiment, mid-circuit measurement included
def circuit_ancilla_measured_first(t, phi, basis):
    """Ancilla read out mid-circuit, THEN the system register. Barriers pin the order."""
    sys_reg, anc_reg = QuantumRegister(N_SYS, "sys"), QuantumRegister(1, "anc")
    qc = QuantumCircuit(sys_reg, anc_reg)
    qc.compose(state_prep_circuit(), qubits=sys_reg, inplace=True)
    qc.h(anc_reg[0])
    qc.append(build_controlled_evolution(HAM, t, "exact"), [anc_reg[0], *sys_reg])
    if phi != 0.0:
        qc.p(phi, anc_reg[0])
    qc.h(anc_reg[0])
    creg = ClassicalRegister(N_SYS + 1, "c")
    qc.add_register(creg)
    qc.measure(anc_reg[0], creg[N_SYS])          # <-- mid-circuit ancilla readout
    qc.barrier()                                 # <-- stops the transpiler reordering
    for j, b in enumerate(basis):
        if b == 0:
            qc.h(sys_reg[j])
        elif b == 1:
            qc.sdg(sys_reg[j]); qc.h(sys_reg[j])
    qc.barrier()
    for j in range(N_SYS):
        qc.measure(sys_reg[j], creg[j])
    return qc


def _counts_vector(counts, shots):
    v = np.zeros(2 ** (N_SYS + 1))
    for k, c in counts.items():
        v[int(k.replace(" ", ""), 2)] = c / shots
    return v


_sim = AerSimulator()
_shots = 200_000
_qc_g = build_shadow_hadamard_circuit(HAM, T_ORD, PHI_IM, BASIS_ORD)
_qc_s = circuit_ancilla_measured_first(T_ORD, PHI_IM, BASIS_ORD)
_vg = _counts_vector(_sim.run(transpile(_qc_g, _sim), shots=_shots, seed_simulator=sub_seed("order-global")).result().get_counts(), _shots)
_vs = _counts_vector(_sim.run(transpile(_qc_s, _sim), shots=_shots, seed_simulator=sub_seed("order-midcircuit")).result().get_counts(), _shots)
_exact = np.array([p_global[a, s] for a in range(2) for s in range(8)])  # int = a*8 + s2*4 + s1*2 + s0

_floor = np.sqrt(2 ** (N_SYS + 1) / (2 * np.pi * _shots))
print(f"total-variation distance to the exact distribution ({_shots:,} shots, noise floor ~{_floor:.4f}):")
print(f"   terminal global measurement : {0.5 * np.abs(_vg - _exact).sum():.5f}")
print(f"   mid-circuit ancilla first   : {0.5 * np.abs(_vs - _exact).sum():.5f}")
check("both measurement orders sample the same distribution",
      0.5 * np.abs(_vg - _exact).sum() < 5 * _floor and 0.5 * np.abs(_vs - _exact).sum() < 5 * _floor)

print("\n=> We use ONE terminal global measurement everywhere below. It is exactly the paper's")
print("   Eq. (C1) and it costs no mid-circuit-measurement overhead on hardware.")

---

# 4 · Running the experiment and keeping every shot

## 4.1 Why we group shots by basis string

The shadow estimator assumes the basis string $b\in\{X,Y,Z\}^n$ is drawn **i.i.d. uniformly per
shot**. A quantum backend, however, runs one *fixed* circuit for many shots. The standard trick:

1. draw all $N$ basis strings at once with NumPy;
2. count how many shots each of the $3^n$ distinct strings got (a multinomial draw);
3. submit one circuit per *distinct* string and keep exactly that many of its shots.

That is statistically identical to per-shot randomisation, and it turns $N$ circuit submissions
into at most $3^n = 27$.

**Implementation note.** Aer runs every circuit in a batch with the same `shots` value, so we
submit `shots = max(counts)` and then truncate each memory list to its own multinomial count. The
truncation is data-independent (shots within a circuit are i.i.d.), hence unbiased; it does mean
Aer *simulates* more raw shots than we keep — about 1.25× at 2000 shots per setting, and worse at
small budgets (≈1.5× at 500) because the multinomial counts fluctuate more. `n_shots` always
counts **kept** records, so your reported shot budget is honest either way.

## 4.2 Getting per-shot data out of Qiskit

Two supported routes. On an **ideal** simulator with the same seed they give byte-identical
records (the checkpoint below proves it); once a noise model is attached the two RNG streams
diverge, so compare distributions, not individual shots:

```python
# route 1 -- backend.run + memory=True   (what we use; oldest, simplest)
result = AerSimulator().run(circuits, shots=N, memory=True, seed_simulator=s).result()
result.get_memory(i)             # -> ['0011', '0010', ...]  one bitstring per shot

# route 2 -- SamplerV2 primitive        (the modern API; use this on IBM hardware)
from qiskit_aer.primitives import SamplerV2
pub = SamplerV2(seed=s).run([circuit], shots=N).result()[0]
pub.data.c.get_bitstrings()      # -> ['0011', '0010', ...]   ('c' = your classical register name)
```

`pub.data.<creg_name>` is a `BitArray`, which also gives you `.get_counts()`, `.array` (a
`(shots, ceil(bits/8))` uint8 array) and `.num_shots`. If you split the ancilla into its own
classical register you get `pub.data.canc` / `pub.data.csys` separately (the names must not
collide with the quantum registers `anc` and `sys`) — convenient for
post-selection.

> ## 🎯 Challenge 3 — per-shot records
>
> **Build** the data layer: the `ShadowRecords` container, `parse_memory`, and
> `run_shadow_hadamard`.
>
> ```
> parse_memory(memory, n)  ->  (outcomes (N,n) of +-1, ancilla (N,) of +-1)
> run_shadow_hadamard(ham, t, phi, n_shots, seed,
>                     prep=None, method="exact", reps=2, backend=None)  ->  ShadowRecords
> ```
>
> **Spec.** For one $(t,\phi)$ setting, produce arrays `bases` $(N,n)$ of ints `0/1/2`, `outcomes`
> $(N,n)$ of $\pm1$, and `ancilla` $(N,)$ of $\pm1$, with bases drawn **i.i.d. uniformly per shot**.
> Store `n_circuits` on the dataclass too — Challenge 5 sums it for the resource table.
> Group shots by distinct basis string so you submit at most $3^n = 27$ circuits instead of $N$,
> request per-shot `memory`, and truncate each memory list to its own multinomial count.
>
> ⚠️ **Trap.** In a memory bitstring the leftmost character is the *highest* classical bit — so the
> ancilla is character 0 and system qubit $j$ is character $n-j$. `parse_memory` must reverse the
> system slice. Get this backwards and the outcome columns stop lining up with the basis columns,
> so **every** system-side estimate is wrong, while $\chi(t)$ — which reads the ancilla only —
> stays perfectly green. A green $\chi$ next to a red $\langle Q\rangle$ in Checkpoint 5 is that
> bug's signature, and it means you should come back here rather than debug §5.
>
> **Keep this schema.** Every estimator in §5 is a one-line NumPy mask over these three arrays. If
> you change the schema, you rewrite everything downstream.
>
> **You will know it works when** Checkpoint 4 is green.

In [ ]:
@dataclass
class ShadowRecords:
    """Every shot of one (t, phi) setting. This schema is the contract for all estimators."""
    t: float
    phi: float
    bases: np.ndarray      # (N, n) ints, 0 = X, 1 = Y, 2 = Z
    outcomes: np.ndarray   # (N, n) +-1 eigenvalue outcomes of the system qubits
    ancilla: np.ndarray    # (N,)   +-1 ancilla outcome  a
    n_circuits: int = 0

    @property
    def n_shots(self) -> int:
        return len(self.ancilla)


def parse_memory(memory: Iterable[str], n: int):
    """Bitstrings -> (+-1 system outcomes (N, n), +-1 ancilla outcomes (N,)).

    Memory string layout (leftmost = highest classical bit):  c[n] c[n-1] ... c[1] c[0]
    so character 0 is the ancilla and character (n - j) is system qubit j.
    """
    # ================================================================== TODO
    # Challenge 3: implement parse_memory().
    # Turn a list of memory bitstrings into (+-1 system outcomes, +-1 ancilla outcomes).
    #
    # Layout (endianness rule #2): the LEFTMOST character is the HIGHEST classical bit, so
    # character 0 is the ancilla and character (n - j) is system qubit j. A '0' outcome is
    # the +1 eigenvalue and a '1' is -1.
    #
    # Work out on paper what parse_memory(['0100', '1011'], n=3) must return before you write
    # anything -- that is exactly what Checkpoint 4 asserts, and both slices are deliberately
    # non-palindromic so a missing reversal fails there rather than in section 5.
    # ==================================================================
    raise NotImplementedError("Challenge 3: implement parse_memory()")


def run_shadow_hadamard(ham, t, phi, n_shots, seed, prep=None,
                        method="exact", reps=2, backend=None) -> ShadowRecords:
    """One (t, phi) setting with per-shot uniformly random local Pauli bases."""
    # ================================================================== TODO
    # Challenge 3: implement run_shadow_hadamard().
    # 0. backend = backend if backend is not None else AerSimulator() -- every caller below
    #    relies on this default. run_time_sweep in section 6.1 uses the same idiom.
    # 1. rng = np.random.default_rng(seed); draw an (n_shots, n) array of ints in {0,1,2}.
    # 2. np.unique(draws, axis=0, return_counts=True) -> the distinct basis strings and how
    #    many shots each got.  This is the multinomial grouping described above.
    # 3. Build one circuit per distinct basis, transpile them together. Forward prep, method
    #    and reps through to build_shadow_hadamard_circuit: section 9 drives this same
    #    function on the Trotter path, and dropping them silently reverts it to the exact
    #    UnitaryGate -- which the Challenge 11 box tells you is meaningless to add noise to.
    # 4. backend.run(circuits, shots=int(counts.max()), memory=True, seed_simulator=...)
    # 5. For circuit i, take result.get_memory(i)[:count_i] -- truncating to that basis'
    #    own multinomial count is what makes the sample exactly i.i.d.-uniform.
    # 6. parse_memory each block, np.tile the basis string, concatenate, and return
    #    ShadowRecords(..., n_circuits=len(circuits)).
    # ==================================================================
    raise NotImplementedError("Challenge 3: implement run_shadow_hadamard()")

### ✅ Checkpoint 4 — bit ordering, uniformity, reproducibility

The first two asserts are a hand-worked example of the memory-string convention: feed in
`"0100"` and `"1011"` and demand the exact $\pm1$ arrays they must produce. Both system slices are
deliberately **non-palindromic**, so an implementation that forgets to reverse the system columns
fails right here rather than three sections later. This is the cheapest
possible insurance against the endianness trap, and it is deterministic — no shots involved.

The remaining asserts confirm that a fixed seed reproduces records bit for bit, that the sampled
bases really are uniform over $\{X,Y,Z\}$, that we kept exactly as many records as we asked for,
and that `n_circuits` was actually recorded — it feeds the resource table in §6.4, and its default
of `0` would otherwise sail through to your submission unnoticed. Judging will re-run your code; if this checkpoint is green, it will get your numbers.

In [ ]:
# --------------------------------------------------- CHECKPOINT 4: bit order and reproducibility
# NB: both system slices below are NON-palindromic on purpose. A parse_memory that forgets to
# reverse the system columns would still pass a palindromic test case such as "0101"/"1000".
sys_out, anc = parse_memory(["0100", "1011"], n=3)
check("memory bitstring -> ancilla outcomes", anc.tolist() == [1, -1], f"{anc.tolist()}")
check("memory bitstring -> system qubit order (reversal included)",
      sys_out[0].tolist() == [1, 1, -1] and sys_out[1].tolist() == [-1, -1, 1],
      f"{sys_out.tolist()}   [an un-reversed slice would give [[-1, 1, 1], [1, -1, -1]]]")

_a = run_shadow_hadamard(HAM, 0.9, PHI_RE, 500, seed=sub_seed("repro"))
_b = run_shadow_hadamard(HAM, 0.9, PHI_RE, 500, seed=sub_seed("repro"))
check("fixed seed -> bit-identical records",
      np.array_equal(_a.bases, _b.bases) and np.array_equal(_a.outcomes, _b.outcomes)
      and np.array_equal(_a.ancilla, _b.ancilla),
      f"bases {np.array_equal(_a.bases, _b.bases)}, "
      f"outcomes {np.array_equal(_a.outcomes, _b.outcomes)}, "
      f"ancilla {np.array_equal(_a.ancilla, _b.ancilla)} "
      f"(bases differ -> the rng is not seeded from `seed`; bases match but outcomes and "
      f"ancilla differ -> seed_simulator was left unpinned)")

_freq = np.array([(_a.bases == c).mean() for c in range(3)])
check("bases are uniform over {X, Y, Z}", np.all(np.abs(_freq - 1 / 3) < 0.05),
      f"frequencies {np.round(_freq, 3).tolist()}")
check("kept records == requested shots", _a.n_shots == 500, f"{_a.n_shots}")
# n_circuits is not used by any estimator, but it feeds the resource table in section 6.4 and
# run_summary.json -- and reporting resources is a non-negotiable judging rule. Leave it at its
# default of 0 and you submit "0 circuits" with an otherwise green notebook.
check("n_circuits recorded", 0 < _a.n_circuits <= _a.n_shots,
      f"{_a.n_circuits} circuits for {_a.n_shots} shots "
      f"(0 means ShadowRecords was built without n_circuits=len(circuits); grouping by "
      f"distinct basis gives {3 ** N_SYS} here, one circuit per shot would give {_a.n_shots} "
      f"-- both are correct, but only the first is resource-efficient)")
print(f"\n({_a.n_circuits} distinct basis circuits submitted for 500 shots)")

Qiskit is migrating from `backend.run` to the **primitives** interface, and on IBM hardware
`SamplerV2` is the route you will actually use. This cell shows the two are interchangeable for our
purpose: same circuit, same seed, same six per-shot bitstrings.

The one API detail worth memorising: `pub.data.<classical register name>` is a `BitArray`, and
`.get_bitstrings()` gives the per-shot list. If you split the classical bits into two registers,
each gets its own field — `pub.data.canc.get_bitstrings()` and `pub.data.csys.get_bitstrings()` —
which makes post-selection on the ancilla a one-liner. Pick names that do not collide with the
*quantum* registers already called `sys` and `anc`, or Qiskit raises
`CircuitError: register name already exists`.

*(This cell is informational. If the import fails in your environment, nothing downstream breaks.)*

In [ ]:
# Optional: the same records via the modern SamplerV2 primitive (skip if the import fails)
_sampler_ok = None
try:
    from qiskit_aer.primitives import SamplerV2

    _sim2 = AerSimulator()
    _qc = transpile(build_shadow_hadamard_circuit(HAM, 0.9, PHI_IM, [0, 1, 2]), _sim2)
    _s = sub_seed("sampler-vs-run")
    _via_sampler = SamplerV2(seed=_s).run([_qc], shots=6).result()[0].data.c.get_bitstrings()
    _via_run = _sim2.run(_qc, shots=6, memory=True, seed_simulator=_s).result().get_memory(0)
    print("SamplerV2      :", _via_sampler)
    print("backend.run    :", _via_run)
    _sampler_ok = (_via_sampler == _via_run)
except Exception as exc:                                        # pragma: no cover
    print("SamplerV2 route unavailable in this environment:", exc)

if _sampler_ok is not None:      # keep the check OUTSIDE the try, or it swallows its own failure
    check("SamplerV2 and backend.run give the same per-shot records", _sampler_ok)

---

# 5 · The three estimators

All three read the **same** `ShadowRecords`. This is the whole point of the paper: one experiment,
many quantities.

| estimator | uses | gives |
|---|---|---|
| `estimate_hadamard_signal` | ancilla column only | $\chi(t) = \operatorname{Tr}[U(t)\rho]$ |
| `estimate_system_observable` | system columns only (unweighted) | $\operatorname{Tr}[O\,\rho^{(I)}(t)]$ — and for conserved $O$, poolable over everything |
| `estimate_joint_observable` | ancilla × system (weighted by $a$) | $\chi_O(t) = \operatorname{Tr}[O\,U(t)\rho]$ |

> ### Pooling rules — get these wrong and your error bars lie
> * Unweighted observables that commute with the **implemented** evolution: pool over **all** $t$
>   and **both** quadratures. ✅  (Exact path: $H$ and $Q$. Trotter path: $Q$ only — see §1.2.)
> * Unweighted **non-commuting** observables: per-$t$ only — but both quadratures may be pooled at
>   that $t$, because $\rho^{(I)}$ is provably $\phi$-independent. ⚠️
> * $\chi_O(t)$: per-$t$ **and** per-quadrature, always. Pooling it across $t$ is a bug. ❌

> ## 🎯 Challenge 4 — the three estimators
>
> This is the heart of the challenge. **Build five functions and a guard**, all reading the
> *same* `ShadowRecords`:
>
> ```
> pauli_terms(obs)                               -> iterator of ({qubit: code}, float coeff)
> pauli_snapshot_values(records, support)        -> np.ndarray of shape (N,)
> estimate_hadamard_signal(rec_re, rec_im)       -> (complex chi, sem_re, sem_im)
> estimate_system_observable(records_list, obs)  -> (float estimate, float sem)
> estimate_joint_observable(rec_re, rec_im, obs) -> (complex chi_O, sem_re, sem_im)
> _check_quadrature_pair(rec_re, rec_im)         -> None, raises on a mismatched pair
> ```
>
> | function | uses | estimates |
> |---|---|---|
> | `pauli_snapshot_values` | one Pauli string, passed as a `{qubit: code}` support dict | the per-shot $\hat P = 3^{w}\prod_j s_j\,\mathbf 1[b_j = P_j]$ |
> | `estimate_hadamard_signal` | ancilla column only | $\chi(t) = \operatorname{Tr}[U(t)\rho]$ |
> | `estimate_system_observable` | system columns, unweighted | $\operatorname{Tr}[O\rho^{(I)}(t)]$ |
> | `estimate_joint_observable` | ancilla $\times$ system, weighted by $a$ | $\chi_O(t) = \operatorname{Tr}[O\,U(t)\rho]$ |
>
> **Spec.** Every estimator returns its standard error alongside the estimate — `sem = std(ddof=1)
> / sqrt(N)` over the per-shot values, never a hand-waved guess. Mind the arities: the two that
> take a quadrature **pair** return *three* values (estimate, sem of the real part, sem of the
> imaginary part), while the pooling one returns *two*. Extend from a single Pauli to a general
> `SparsePauliOp` by linearity over `pauli_terms`.
>
> ⚠️ **Traps.** (i) The identity Pauli has $\hat P \equiv 1$, so its ancilla-weighted average is just
> $\chi(t)$ — handle observables with an identity component accordingly. (ii) Read the pooling
> rules above and obey them; pooling $\chi_O$ across times is the most common way to produce
> confident nonsense. (iii) Both `estimate_*` functions that take a quadrature pair should *guard*
> against being handed swapped or mismatched records — a silent failure mode that produces a
> plausible-looking spectrum with the imaginary part inverted.
>
> **You will know it works when** Checkpoint 5 is green (nine $5\sigma$ gates).

In [ ]:
def pauli_terms(obs: SparsePauliOp):
    """Yield (support {qubit: code}, real coefficient) for each Pauli term of `obs`."""
    for label, coeff in obs.simplify().to_list():
        support = {q: PAULI_CODES[ch] for q, ch in enumerate(label[::-1]) if ch != "I"}
        yield support, np.real_if_close(coeff)


def pauli_snapshot_values(records: ShadowRecords, support: Mapping[int, int]) -> np.ndarray:
    """Per-shot local-shadow estimates  hat{P}_s  of ONE Pauli string.

        hat{P} = 3^w * prod_{j in supp} s_j * 1[b_j == P_j]        (identity -> all ones)
    """
    # ================================================================== TODO
    # Challenge 4: implement pauli_snapshot_values().
    # Per-shot single-Pauli shadow estimator:
    #     hat{P} = 3^w * prod_{j in supp} s_j * 1[b_j == P_j]
    #
    # `support` is a {qubit: code} dict with code in {0:X, 1:Y, 2:Z}; w = len(support).
    # The empty support is the identity Pauli, whose estimator is identically 1.
    #
    # Vectorise it -- no Python loop over shots:
    #     match = np.all(records.bases[:, qubits] == codes, axis=1)
    #     prod  = np.prod(records.outcomes[:, qubits], axis=1)
    # ==================================================================
    raise NotImplementedError("Challenge 4: implement pauli_snapshot_values()")


def _check_quadrature_pair(rec_re: ShadowRecords, rec_im: ShadowRecords) -> None:
    """Guard against swapped or mismatched quadrature records -- a silent-nonsense trap."""
    if not np.isclose(rec_re.phi, PHI_RE) or not np.isclose(rec_im.phi, PHI_IM):
        raise ValueError(f"expected (phi=0, phi=-pi/2), got ({rec_re.phi}, {rec_im.phi})")
    if not np.isclose(rec_re.t, rec_im.t):
        raise ValueError(f"quadrature records at different t: {rec_re.t} vs {rec_im.t}")


# ---------------------------------------------------------------------- 1) ancilla only
def estimate_hadamard_signal(rec_re: ShadowRecords, rec_im: ShadowRecords):
    """chi(t) = Tr[U(t) rho] from the ancilla marginal.  Returns (chi, sem_re, sem_im)."""
    _check_quadrature_pair(rec_re, rec_im)
    re, im = np.mean(rec_re.ancilla), np.mean(rec_im.ancilla)
    sem_re = np.std(rec_re.ancilla, ddof=1) / np.sqrt(rec_re.n_shots)
    sem_im = np.std(rec_im.ancilla, ddof=1) / np.sqrt(rec_im.n_shots)
    return re + 1j * im, float(sem_re), float(sem_im)


# ------------------------------------------------------- 2) unweighted shadows  ->  rho^(I)
def estimate_system_observable(records_list: Sequence[ShadowRecords], obs: SparsePauliOp):
    """<O> under rho^(I), pooling every record set given.  Returns (estimate, sem)."""
    # ================================================================== TODO
    # Challenge 4: implement estimate_system_observable().
    # <O> under rho^(I), pooling every ShadowRecords object passed in.
    #
    # For each (support, coeff) from pauli_terms(obs): concatenate that Pauli's per-shot
    # values across ALL records, scale by coeff, and accumulate.  Then return
    #     (mean of the accumulated per-shot values, std(ddof=1)/sqrt(N)).
    #
    # Returns a 2-tuple (estimate, sem) -- unlike the two quadrature-pair estimators.
    # Pooling rule: legitimate across times only for observables that commute with the
    # implemented evolution; always legitimate across the two quadratures.
    # ==================================================================
    raise NotImplementedError("Challenge 4: implement estimate_system_observable()")


# ------------------------------------------- 3) ancilla-weighted shadows  ->  chi_O(t)
def estimate_joint_observable(rec_re: ShadowRecords, rec_im: ShadowRecords, obs: SparsePauliOp):
    """chi_O(t) = Tr[O U(t) rho] from a-weighted shadows.  Returns (chi_O, sem_re, sem_im)."""
    # ================================================================== TODO
    # Challenge 4: implement estimate_joint_observable().
    # chi_O(t) = Tr[O U(t) rho] from ANCILLA-WEIGHTED shadows.
    #
    # E[a * hat{P}]_phi = Re[e^{i phi} Tr(P U rho)], so with rec_re at phi=0 and rec_im at
    # phi=-pi/2 the real and imaginary parts come from the two record sets separately.
    #
    # 1. call _check_quadrature_pair(rec_re, rec_im) first -- a swapped pair is silent nonsense
    # 2. build the per-shot observable value for each record set by summing
    #    coeff * pauli_snapshot_values(rec, support) over pauli_terms(obs)
    # 3. multiply each by that record set's ancilla column (+-1)
    # 4. return (mean_re + 1j*mean_im, sem_re, sem_im)
    #
    # Do NOT use in-place '+=' on the accumulator: if a coefficient stays complex, in-place
    # addition raises instead of upcasting.
    # ==================================================================
    raise NotImplementedError("Challenge 4: implement estimate_joint_observable()")

### ✅ Checkpoint 5 — all three estimators against exact references

Nine asserts at a single time point $t = 0.9$ with 6000 shots per quadrature, each a $5\sigma$
gate: both quadratures of $\chi$, the two conserved observables, the non-conserved $Z_0$, and both
quadratures of two different joint observables.

Why $5\sigma$ and not $2\sigma$? Because with fixed seeds these tests must never flake — a
$2\sigma$ gate fails roughly one run in twenty *by construction*, which would train you to ignore
red cells. A $5\sigma$ gate that fails is a bug, full stop.

⚠️ **The trap this checkpoint is built around** is the $Z_0$ line. $Z_0$ does not commute with $H$,
so its unweighted shadow average estimates $\operatorname{Tr}[Z_0\rho^{(I)}(t)]$ — the
*time-dependent marginal* — and **not** $\langle Z_0\rangle_\rho$. Compare against the wrong
reference and you will see a large, seemingly systematic error and go hunting for a bug that is not
there. The printout at the end shows both numbers side by side.

In [ ]:
# --------------------------------- CHECKPOINT 5: all three estimators, one time point, 5 sigma
t_c = 0.9
rec_re = run_shadow_hadamard(HAM, t_c, PHI_RE, 6000, seed=sub_seed("cp5-re"))
rec_im = run_shadow_hadamard(HAM, t_c, PHI_IM, 6000, seed=sub_seed("cp5-im"))

chi_hat, s_re, s_im = estimate_hadamard_signal(rec_re, rec_im)
chi_ref = exact_chi(HAM, PSI, [t_c])[0]
check("chi(t) real part", *sigma_gate(chi_hat.real, chi_ref.real, s_re))
check("chi(t) imag part", *sigma_gate(chi_hat.imag, chi_ref.imag, s_im))

h_hat, h_sem = estimate_system_observable([rec_re, rec_im], HAM)
check("<H> from unweighted shadows", *sigma_gate(h_hat, H_EXPECT_EXACT, h_sem))

q_hat, q_sem = estimate_system_observable([rec_re, rec_im], CHARGE)
check("<Q> from unweighted shadows", *sigma_gate(q_hat, Q_EXPECT_EXACT, q_sem))

z0_hat, z0_sem = estimate_system_observable([rec_re, rec_im], Z0_OBS)
z0_ref = exact_system_marginal_expectation(HAM, PSI, Z0_OBS, t_c)
check("<Z0> vs the rho^(I) reference (NOT vs <Z0>_rho!)", *sigma_gate(z0_hat, z0_ref, z0_sem))

for name, obs in (("Q", CHARGE), ("Z0", Z0_OBS)):
    est, cr, ci = estimate_joint_observable(rec_re, rec_im, obs)
    ref = exact_chi_O(HAM, PSI, obs, [t_c])[0]
    check(f"chi_{name}(t) real", *sigma_gate(est.real, ref.real, cr))
    check(f"chi_{name}(t) imag", *sigma_gate(est.imag, ref.imag, ci))

print(f"\nNote how <Z0>_rho^(I)(t) = {z0_ref:+.4f} differs from <Z0>_rho = "
      f"{float(np.real(PSI.conj() @ Z0_OBS.to_matrix() @ PSI)):+.4f}: Z0 is NOT conserved,")
print("so its unweighted shadow average tracks the time-dependent marginal, not the input state.")

---

# 6 · PART A — the fundamental goal

## 6.1 One sweep, everything at once

`run_time_sweep` walks the time grid, runs both quadratures at each $t$, and immediately extracts
$\chi(t)$ plus any $\chi_O(t)$ you asked for. **Crucially it keeps every `ShadowRecords` object**,
so later sections can mine the same shots for new quantities without touching the simulator again.

We request three joint observables:

* $Q$ (conserved) → the symmetry-label channel for Part B;
* $H$ (conserved) → the Krylov channel for the bonus in §8;
* $Z_0$ (**not** conserved) → the fundamental goal's "one more observable" requirement.

> ## 🎯 Challenge 5 — the time sweep *(given complete — read it, do not rewrite it)*
>
> **Read** `run_time_sweep(...) -> SweepResult`: walk the time grid, run both quadratures at each
> $t$, and extract $\chi(t)$ plus every requested $\chi_O(t)$ with uncertainties.
>
> **Spec.** Derive per-setting seeds from one master seed with `np.random.SeedSequence(seed).spawn`
> so the whole sweep is reproducible and the streams are independent. Accept a `backend` argument
> so the same function can later be pointed at a noisy simulator. Count circuits and kept shots.
>
> **The one design decision that matters:** `SweepResult.records` **keeps every `ShadowRecords`
> object**. It is tempting to throw them away once you have $\chi$ and $\chi_O$ — don't. Sections
> 6, 7 and 8 all mine those same records for quantities we have not thought of yet, and that
> re-mining *is* the point of the paper. A pipeline that discards its shot records has thrown away
> the contribution.

In [ ]:
@dataclass
class SweepResult:
    ts: np.ndarray
    chi: np.ndarray                 # complex chi(t) from the ancilla
    chi_sem: np.ndarray             # (N_t, 2) sem of Re / Im
    chi_obs: dict                   # name -> complex chi_O(t)
    chi_obs_sem: dict               # name -> (N_t, 2)
    records: list                   # [(rec_re, rec_im), ...]  <-- keep them, they are the asset
    total_shots: int = 0
    total_circuits: int = 0


def run_time_sweep(ham, ts, shots_per_setting, seed,
                   joint_observables=None, prep=None,
                   method="exact", reps=2, backend=None, verbose=True) -> SweepResult:
    """Both quadratures at every t; estimate chi(t) and every requested chi_O(t).

    Pass backend=AerSimulator(noise_model=...) to run the whole sweep noisily.
    """
    backend = backend if backend is not None else AerSimulator()
    joint_observables = joint_observables or {}
    seeds = np.random.SeedSequence(seed).spawn(2 * len(ts))     # reproducible, independent streams

    chi, chi_sem, records = [], [], []
    chi_obs = {k: [] for k in joint_observables}
    chi_obs_sem = {k: [] for k in joint_observables}
    tot_shots = tot_circ = 0

    for i, t in enumerate(ts):
        rec_re = run_shadow_hadamard(ham, t, PHI_RE, shots_per_setting,
                                     seed=int(seeds[2 * i].generate_state(1)[0]) % (2 ** 31),
                                     prep=prep, method=method, reps=reps, backend=backend)
        rec_im = run_shadow_hadamard(ham, t, PHI_IM, shots_per_setting,
                                     seed=int(seeds[2 * i + 1].generate_state(1)[0]) % (2 ** 31),
                                     prep=prep, method=method, reps=reps, backend=backend)

        c, sr, si = estimate_hadamard_signal(rec_re, rec_im)
        chi.append(c); chi_sem.append((sr, si))
        for name, obs in joint_observables.items():
            co, csr, csi = estimate_joint_observable(rec_re, rec_im, obs)
            chi_obs[name].append(co); chi_obs_sem[name].append((csr, csi))

        records.append((rec_re, rec_im))
        tot_shots += rec_re.n_shots + rec_im.n_shots
        tot_circ += rec_re.n_circuits + rec_im.n_circuits
        if verbose and (i % 8 == 0 or i == len(ts) - 1):
            print(f"  t = {t:6.2f}  ({i + 1:3d}/{len(ts)})   chi = {c:+.3f}")

    return SweepResult(np.asarray(ts), np.array(chi), np.array(chi_sem),
                       {k: np.array(v) for k, v in chi_obs.items()},
                       {k: np.array(v) for k, v in chi_obs_sem.items()},
                       records, tot_shots, tot_circ)

This is the experiment. One call, and it produces everything Parts A and B are graded on.

Watch what comes out of it: `CHI`, `CHI_Q`, `CHI_H`, `CHI_Z0`, **and** `ALL_RECS` — $2N_t$ record
objects (128 at the official budget) holding every single shot. Sections 6, 7 and 8 then *re-mine
that one variable* rather than going back to the simulator. That is the paper's thesis, made
operational.

Expected cost: 3 456 circuits, 256 000 shots, about 50 s here and a couple of minutes on Colab.

*(Two later sections do run their own jobs, deliberately: the $N^{-1/2}$ scaling study in §6.3
needs independent small runs at several shot counts, and the §9 noise bonus needs a second backend.
Neither reuses this sweep, and neither feeds Part B.)*

In [ ]:
JOINT_OBS = {"Q": CHARGE, "H": HAM, "Z0": Z0_OBS}

print(f"Running the sweep: {N_TIMES} times x 2 quadratures x {SHOTS} shots ...")
_t0 = time.time()
SWEEP = run_time_sweep(HAM, TS, SHOTS, SEED, joint_observables=JOINT_OBS)
WALL = time.time() - _t0

CHI      = SWEEP.chi
CHI_Q    = SWEEP.chi_obs["Q"]
CHI_H    = SWEEP.chi_obs["H"]
CHI_Z0   = SWEEP.chi_obs["Z0"]
ALL_RECS = [r for pair in SWEEP.records for r in pair]

print(f"\ndone in {WALL:.1f} s")
print(f"{SWEEP.total_circuits:,} circuits, {SWEEP.total_shots:,} kept shots, "
      f"{len(ALL_RECS)} record objects retained")

## 6.2 Validating every estimate against the exact reference

The sweep is done and the simulator is behind us. Everything from here is post-processing: we
compare each estimate with the exact reference, in units of its own predicted standard error.

> ## 🎯 Challenge 6 — validate everything, then prove it is shot-noise limited *(given complete — read it, do not rewrite it)*
>
> Part A is only finished when you can *demonstrate* correctness, not assert it. Over §6.2–6.4 the
> evidence is produced for you — read it, and understand why each item is the right test:
>
> 1. $\chi(t)$ and $\chi_Q(t)$ against exact references — rms error, mean predicted sem, and worst
>    deviation in units of $\sigma$. The rms error should land close to the predicted sem; if it is
>    much larger you have a bias, if much smaller your error bars are inflated.
> 2. $\langle H\rangle$ and $\langle Q\rangle$ pooled over **all** records, each within a few
>    $\sigma$ — and note $\langle Q\rangle$ comes out about $1.6\times$ tighter, the direction the
>    $3^{w}$ rule predicts.
> 3. A non-conserved observable at one time against the $\rho^{(I)}$ reference.
> 4. The $N^{-1/2}$ scaling study.
>
> **Reference calibration** (official budget, seed 2026): $\chi(t)$ rms error $\approx 0.026$,
> $\langle H\rangle$ within $\pm0.01$, $\langle Q\rangle$ within $\pm0.006$, fitted slope
> $\approx -0.52$. Match that *ballpark* — not decimal for decimal, since your seeds will differ.
>
> **You will know it works when** Checkpoint 6 is green: eight asserts, one per acceptance criterion.

In [ ]:
CHI_REF    = exact_chi(HAM, PSI, TS)
CHI_Q_REF  = exact_chi_O(HAM, PSI, CHARGE, TS)
CHI_H_REF  = exact_chi_O(HAM, PSI, HAM, TS)
CHI_Z0_REF = exact_chi_O(HAM, PSI, Z0_OBS, TS)

err   = np.abs(CHI - CHI_REF)
sem   = np.hypot(SWEEP.chi_sem[:, 0], SWEEP.chi_sem[:, 1])
err_q = np.abs(CHI_Q - CHI_Q_REF)
sem_q = np.hypot(SWEEP.chi_obs_sem["Q"][:, 0], SWEEP.chi_obs_sem["Q"][:, 1])

print("Signal quality")
print(f"  chi(t)   : rms error {np.sqrt(np.mean(err ** 2)):.4f}   "
      f"mean predicted sem {np.mean(sem):.4f}   max |err|/sem {np.max(err / sem):.2f}")
print(f"  chi_Q(t) : rms error {np.sqrt(np.mean(err_q ** 2)):.4f}   "
      f"mean predicted sem {np.mean(sem_q):.4f}")

# conserved observables: pool EVERY record (all times, both quadratures)
H_HAT, H_SEM = estimate_system_observable(ALL_RECS, HAM)
Q_HAT, Q_SEM = estimate_system_observable(ALL_RECS, CHARGE)
print(f"\nPooled over all {len(ALL_RECS)} record sets ({SWEEP.total_shots:,} shots):")
print(f"  <H> = {H_HAT:+.4f} +- {H_SEM:.4f}   (exact {H_EXPECT_EXACT:+.4f}, "
      f"{abs(H_HAT - H_EXPECT_EXACT) / H_SEM:.1f} sigma)")
print(f"  <Q> = {Q_HAT:+.4f} +- {Q_SEM:.4f}   (exact {Q_EXPECT_EXACT:+.4f}, "
      f"{abs(Q_HAT - Q_EXPECT_EXACT) / Q_SEM:.1f} sigma)")
print(f"  [note <Q> is {H_SEM / Q_SEM:.1f}x tighter than <H>: Q is all weight-1 Paulis, "
      f"H has weight-2 terms with 3^2 single-shot second moment]")

# non-conserved observable: per-time only, checked against the rho^(I) reference
PROBE_IDX = N_TIMES // 3   # avoid the name _i: IPython reserves it for the previous input
Z0_HAT, Z0_SEM = estimate_system_observable(list(SWEEP.records[PROBE_IDX]), Z0_OBS)
Z0_REF = exact_system_marginal_expectation(HAM, PSI, Z0_OBS, TS[PROBE_IDX])
print(f"\n  <Z0>_rho^(I)(t={TS[PROBE_IDX]:.1f}) = {Z0_HAT:+.4f} +- {Z0_SEM:.4f}   (exact {Z0_REF:+.4f}, "
      f"{abs(Z0_HAT - Z0_REF) / Z0_SEM:.1f} sigma)")

### Seeing the conservation law directly

The pooling rule above is abstract; this figure makes it concrete, and it is the single most
useful diagnostic plot in Part A.

**Left panel** — $\langle Q\rangle$ estimated *independently at each time point*. Because
$[Q,H]=0$, every one of these estimates targets the same number, so the points must scatter around
a flat line at $\langle Q\rangle_\rho$ with no drift. The final assert checks exactly that, at
$5\sigma$, at every time. A tilt or a wobble here means your evolution is not conserving what you
think it is — a much sharper test than looking at the pooled average alone.

**Right panel** — the same treatment for $Z_0$, which is *not* conserved. It starts exactly *on* the dotted line at
$t=0$ — where $\rho^{(I)}(0) = \rho$, so the two references coincide — then walks away from it and
tracks $\operatorname{Tr}[Z_0\rho^{(I)}(t)]$ instead. The input-state value is right at one point
and wrong everywhere else. Put these two panels side by side in your report: they are the clearest
possible evidence that you understand which quantities may be pooled.

In [ ]:
# ---- per-time conservation check: <Q>(t) must be flat -------------------------------------
q_t = np.array([estimate_system_observable(list(pair), CHARGE) for pair in SWEEP.records])
z0_t = np.array([estimate_system_observable(list(pair), Z0_OBS) for pair in SWEEP.records])
z0_t_ref = np.array([exact_system_marginal_expectation(HAM, PSI, Z0_OBS, t) for t in TS])

fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.3))
ax = axes[0]
ax.errorbar(TS, q_t[:, 0], yerr=q_t[:, 1], fmt="o", ms=3.2, lw=1, color=COL["aqua"],
            label=r"$\langle Q\rangle_{\rho^{(I)}(t)}$ per time")
ax.axhline(Q_EXPECT_EXACT, color=COL["muted"], lw=1.4, label=r"exact $\langle Q\rangle_\rho$")
ax.set_xlabel("t"); ax.set_ylabel(r"$\langle Q\rangle$")
ax.set_title("Conserved: flat in $t$, so all records pool", fontsize=10)
ax.legend(frameon=False, fontsize=8)

ax = axes[1]
ax.plot(TS, z0_t_ref, color=COL["muted"], lw=1.4, label=r"exact $\mathrm{Tr}[Z_0\rho^{(I)}(t)]$")
ax.errorbar(TS, z0_t[:, 0], yerr=z0_t[:, 1], fmt="o", ms=3.2, lw=1, color=COL["orange"],
            label="unweighted shadows")
ax.axhline(float(np.real(PSI.conj() @ Z0_OBS.to_matrix() @ PSI)), color=COL["violet"],
           ls=":", lw=1.4, label=r"$\langle Z_0\rangle_\rho$ (wrong reference!)")
ax.set_xlabel("t"); ax.set_ylabel(r"$\langle Z_0\rangle$")
ax.set_title("Not conserved: drifts, per-$t$ only", fontsize=10)
ax.legend(frameon=False, fontsize=8)

for a in axes:
    style_axes(a)
fig.tight_layout()
show(fig, "06_conservation")

_zs = np.abs(q_t[:, 0] - Q_EXPECT_EXACT) / q_t[:, 1]
check("per-time <Q> is constant within 5 sigma everywhere", np.max(_zs) < 5.0,
      f"max {np.max(_zs):.1f} sigma over {N_TIMES} times", budget_sensitive=True)

## 6.3 The four-panel validation figure

Before the figure, one more piece of evidence has to be generated: the **shot-noise scaling study**.

The logic is worth spelling out, because it is the only test here that can distinguish a
*statistical* error from a *systematic* one. Every other check compares an estimate against a
reference within its error bar — but an estimator with a small constant bias will pass those at low
shot counts and only fail once the error bars shrink below the bias. Scaling exposes it directly:
an unbiased estimator's error falls as $N^{-1/2}$ forever, while a biased one flattens onto a floor.

So the cell below runs the experiment at three shot counts, $N = 128$, $512$ and $2048$, with **24
independent seeds each**, averages $|\hat\chi - \chi|$ at a single time, and fits a straight line
through the log-log points. Why 24 seeds rather than a handful? Because the fitted slope is itself
a random variable: with 8 repeats its standard deviation is about $0.10$, so a $\pm0.12$ acceptance
gate would fail roughly one run in five purely by chance. With 24 it drops to about $0.06$ and the
test becomes meaningful rather than a coin flip.

These runs are deliberately *independent* of the main sweep — different shot counts, fresh seeds —
because reusing subsets of one sample would correlate the three points and flatter the fit.

In [ ]:
# ---- shot-noise scaling study (independent small runs at one time) --------------------------
T_PROBE = 0.9
CHI_PROBE = exact_chi(HAM, PSI, [T_PROBE])[0]
SCALE_NS = (128, 512, 2048)
N_REPEATS = 24        # 8 repeats makes the fitted slope itself noisy (sd ~ 0.10)

scale_err = []
for n_sh in SCALE_NS:
    errs = [abs(estimate_hadamard_signal(
                run_shadow_hadamard(HAM, T_PROBE, PHI_RE, n_sh, seed=sub_seed(f"scale-re-{n_sh}", s)),
                run_shadow_hadamard(HAM, T_PROBE, PHI_IM, n_sh, seed=sub_seed(f"scale-im-{n_sh}", s)))[0]
                - CHI_PROBE) for s in range(N_REPEATS)]
    scale_err.append(float(np.mean(errs)))
    print(f"  N = {n_sh:5d}   mean |chi_hat - chi| = {scale_err[-1]:.4f}")

SCALE_SLOPE = float(np.polyfit(np.log(SCALE_NS), np.log(scale_err), 1)[0])
print(f"\n  fitted scaling ~ N^{SCALE_SLOPE:+.2f}   (theory: -0.50)")
check("shot-noise scaling is N^-1/2", abs(SCALE_SLOPE + 0.5) < 0.15, f"slope {SCALE_SLOPE:+.3f}")

### The Part-A headline figure

Four panels, and each one is evidence for a different acceptance criterion — this is the figure to
put in your report.

1. **Re $\chi(t)$** — the ancilla alone, $\phi = 0$. Standard Hadamard test territory.
2. **Im $\chi(t)$** — the same circuits at $\phi = -\pi/2$. Together these two panels prove your
   quadrature convention is right; if the second one is mirrored, re-read §1.1.
3. **Re $\chi_Q(t)$** — the ancilla-**weighted** shadows. *This panel does not exist in the standard
   Hadamard test.* It is the joint observable, obtained from the same shots as panels 1–2 by
   multiplying each snapshot by $a = \pm1$. Note the error bars are visibly larger: shadow
   estimators pay a $3^{w}$ variance premium.
4. **Error vs shots** — the log-log line from the cell above, with an $N^{-1/2}$ guide. A slope of
   $-1/2$ says you are shot-noise limited and unbiased. A visible floor would say you have a
   systematic error (Trotter bias, wrong reference, a leaked normalisation).

In [ ]:
tf = np.linspace(TS[0], TS[-1], 600)
chi_fine   = exact_chi(HAM, PSI, tf)
chi_q_fine = exact_chi_O(HAM, PSI, CHARGE, tf)

fig, axes = plt.subplots(2, 2, figsize=(10.6, 6.6))

ax = axes[0, 0]
ax.plot(tf, chi_fine.real, color=COL["muted"], lw=1.4, label="exact")
ax.errorbar(TS, CHI.real, yerr=SWEEP.chi_sem[:, 0], fmt="o", ms=3.4, lw=1,
            color=COL["blue"], label="ancilla estimate")
ax.set_xlabel("t"); ax.set_ylabel(r"Re $\chi(t)$")
ax.set_title("Hadamard signal, real quadrature ($\\phi=0$)", fontsize=10)
ax.legend(frameon=False, fontsize=8)

ax = axes[0, 1]
ax.plot(tf, chi_fine.imag, color=COL["muted"], lw=1.4, label="exact")
ax.errorbar(TS, CHI.imag, yerr=SWEEP.chi_sem[:, 1], fmt="o", ms=3.4, lw=1,
            color=COL["blue"], label=r"ancilla, $\phi=-\pi/2$")
ax.set_xlabel("t"); ax.set_ylabel(r"Im $\chi(t)$")
ax.set_title("Hadamard signal, imaginary quadrature", fontsize=10)
ax.legend(frameon=False, fontsize=8)

ax = axes[1, 0]
ax.plot(tf, chi_q_fine.real, color=COL["muted"], lw=1.4, label=r"exact Re $\chi_Q$")
ax.errorbar(TS, CHI_Q.real, yerr=SWEEP.chi_obs_sem["Q"][:, 0], fmt="o", ms=3.4, lw=1,
            color=COL["orange"], label="ancilla-weighted shadows")
ax.set_xlabel("t"); ax.set_ylabel(r"Re $\chi_Q(t)$")
ax.set_title(r"Joint observable $\chi_Q(t)=\mathrm{Tr}[Q\,U(t)\rho]$", fontsize=10)
ax.legend(frameon=False, fontsize=8)

ax = axes[1, 1]
ax.loglog(SCALE_NS, scale_err, "o-", color=COL["aqua"], ms=5, lw=1.6,
          label=r"mean $|\hat\chi-\chi|$")
ax.loglog(SCALE_NS, scale_err[0] * (np.array(SCALE_NS) / SCALE_NS[0]) ** -0.5, "--",
          color=COL["muted"], lw=1.2, label=r"$N^{-1/2}$")
ax.set_xlabel("shots per quadrature"); ax.set_ylabel("error")
ax.set_title(f"Shot-noise scaling at t = {T_PROBE}", fontsize=10)
ax.legend(frameon=False, fontsize=8)

for a in axes.flat:
    style_axes(a)
fig.suptitle("Fundamental goal: shadow-enhanced Hadamard test vs. exact reference",
             fontsize=12, color=COL["ink"])
fig.tight_layout(rect=(0, 0, 1, 0.96))
show(fig, "06_fundamental_validation")

## 6.4 ✅ Checkpoint 6 — acceptance checklist and resource accounting

Eight asserts, one per acceptance criterion, forming the Part-A grading contract. Read them as a
checklist of what "done" means rather than as a pass/fail gate:

* **6.1–6.2** — $\chi(t)$ agrees with exact everywhere, *and* its rms error is consistent with the
  sem the estimator predicted for itself. The second one is subtler than the first: an estimator
  can be accurate while systematically mis-reporting its own uncertainty, and a submission whose
  error bars are twice too small is worse than one that is slightly less precise and honest. The
  ratio should land near 1 (we require $0.5$–$1.6$).
* **6.3–6.5** — the two conserved observables from pooled shadows and the non-conserved one
  against the correct $\rho^{(I)}$ reference.
* **6.6** — the joint observable $\chi_Q(t)$, the quantity that does not exist in a standard
  Hadamard test.
* **6.7** — the $N^{-1/2}$ scaling, i.e. unbiasedness.
* **6.8** — that all of this came from *shared* records rather than separate experiments, which is
  the entire point of the paper.

**Reference calibration** (official budget, seed 2026): $\chi(t)$ rms error $\approx 0.026$,
$\langle H\rangle$ within $\pm0.01$, $\langle Q\rangle$ within $\pm0.006$, fitted scaling slope
$\approx -0.52$. Match that *ballpark*, not decimal for decimal — your seeds will differ and so
will your last two digits.

In [ ]:
# ---------------------------------------------- CHECKPOINT 6: the fundamental goal, item by item
check("6.1  chi(t) agrees with exact everywhere (5 sigma)",
      np.max(err / sem) < 5.0, f"max {np.max(err / sem):.2f} sigma over {N_TIMES} times", budget_sensitive=True)
check("6.2  chi(t) rms error consistent with predicted sem",
      0.5 < np.sqrt(np.mean(err ** 2)) / np.mean(sem) < 1.6,
      f"rms/sem = {np.sqrt(np.mean(err ** 2)) / np.mean(sem):.2f}", budget_sensitive=True)
check("6.3  <H> from pooled shadows", *sigma_gate(H_HAT, H_EXPECT_EXACT, H_SEM), budget_sensitive=True)
check("6.4  <Q> from pooled shadows", *sigma_gate(Q_HAT, Q_EXPECT_EXACT, Q_SEM), budget_sensitive=True)
check("6.5  non-conserved <Z0> vs the rho^(I) reference", *sigma_gate(Z0_HAT, Z0_REF, Z0_SEM), budget_sensitive=True)
check("6.6  joint chi_Q(t) agrees with exact (5 sigma)",
      np.max(err_q / sem_q) < 5.0, f"max {np.max(err_q / sem_q):.2f} sigma", budget_sensitive=True)
check("6.7  error scales as N^-1/2", abs(SCALE_SLOPE + 0.5) < 0.15, f"{SCALE_SLOPE:+.3f}", budget_sensitive=True)
check("6.8  several observables from SHARED records",
      len(JOINT_OBS) >= 3 and len(ALL_RECS) == 2 * N_TIMES,
      f"{len(JOINT_OBS)} joint observables + <H>,<Q>,<Z0> from {len(ALL_RECS)} record sets", budget_sensitive=True)

### Resource accounting — a required deliverable

Judging weights *resource efficiency* at 20 %, so this table is not optional bookkeeping. It
reports transpiled depth and one- and two-qubit gate counts for the exact path and for three
Trotter settings, plus the experiment totals.

Read it with the §3.1 caveat in mind: at $n=3$ the exact path wins on CX count and depth (though
not on one-qubit gates), and that ordering inverts as $n$ grows. Quote both rows, and say which one
you would run on hardware and why.

The last line is the one to be proud of: seven distinct physical quantities, all from the same
256 000 shots.

In [ ]:
# ------------------------------------------------------------------ resource table (deliverable)
def resource_row(method, reps=2, basis_gates=("rz", "sx", "x", "cx")):
    qc = build_shadow_hadamard_circuit(HAM, 0.9, PHI_RE, [0, 1, 2], method=method, reps=reps)
    tq = transpile(qc, basis_gates=list(basis_gates), optimization_level=1, seed_transpiler=0)
    ops = tq.count_ops()
    one_q = sum(v for k, v in ops.items() if k in ("rz", "sx", "x", "h", "s", "sdg", "ry", "rx"))
    return dict(circuit=method if method == "exact" else f"trotter reps={reps}",
                depth=tq.depth(), one_q=one_q, two_q=ops.get("cx", 0))

rows = [resource_row("exact"), resource_row("trotter", 1),
        resource_row("trotter", 2), resource_row("trotter", 4)]

print("Per-circuit transpiled cost  (basis {rz, sx, x, cx}, optimization_level=1)\n")
print(f"{'circuit':>16} | {'depth':>6} | {'1q gates':>9} | {'2q gates (cx)':>14}")
print("-" * 56)
for r in rows:
    print(f"{r['circuit']:>16} | {r['depth']:>6} | {r['one_q']:>9} | {r['two_q']:>14}")

print(f"\nExperiment totals")
print(f"  distinct circuits submitted : {SWEEP.total_circuits:,}")
print(f"  shots kept                  : {SWEEP.total_shots:,}")
print(f"  wall clock (sweep only)     : {WALL:.1f} s")
print(f"  quantities extracted        : chi(t), chi_Q(t), chi_H(t), chi_Z0(t), <H>, <Q>, "
      f"<Z0>_rho^(I)(t)  -- from the SAME shots")

> ### 🧪 Going further — fundamental-goal extensions the rubric rewards
>
> Challenges 1–6 got you a correct baseline. These are the moves that turn a correct submission into a strong one; none of them is implemented above.
> 1. **Redo the whole sweep with `method="trotter"`** and quantify the Trotter bias against shot
>    noise: at which `reps` does the bias drop below the statistical error at your budget?
>    Watch the pooling rule while you do it — $Q$ still commutes with every Trotter factor, but
>    $H$ does not, so the pooled $\langle H\rangle$ acquires a systematic offset (§1.2). Measuring
>    that offset *is* the exercise.
> 2. **Variance budgeting.** Predict the sem of $\langle H \rangle$ analytically from the $3^{w}$
>    rule and compare with the measured one, term by term.
> 3. **Derandomised / biased Pauli sampling.** You know in advance which Paulis you care about
>    ($H$, $Q$, $Z_0$). Bias the basis distribution towards them and show a variance reduction at
>    fixed shot count — then check you have not biased the *estimator* (importance weights!).
> 4. **Correlation.** $\hat\chi$ and $\hat\chi_Q$ come from the same shots, so their errors are
>    correlated (measured $\approx +0.5$ here). Estimate that correlation and propagate it — the
>    bootstrap in §7.3 ignores it and is therefore conservative.

---

# 7 · PART B — the advanced goal

## 7.0 Choose exactly ONE track

| Track | Idea | Status here |
|---|---|---|
| **A — Symmetry-resolved spectral analyzer** *(flagship)* | recover $(E_k, p_k, \hat q_k)$ with uncertainties from $\{\hat\chi,\hat\chi_Q\}$ | **fully implemented below** |
| **B — Dynamics A/B tester** | add an *anti*-controlled $W$ so the ancilla estimates $\operatorname{Tr}[W^\dagger U\rho]$; shadows then localise *which observables* two dynamics disagree on. The "difference of evolved states" $\rho^{(X)}$ needs an $X$-basis ancilla readout — i.e. **omit the final Hadamard** (paper App. B, E) | scaffold in §7.5 |
| **C — Eigenstate detective** | for an eigenstate $|\chi(t)|\equiv1$ and *every* $\rho^{(I)}$ observable is time-independent; build quantitative eigenstateness witnesses and calibrate detection power vs. shots | scaffold in §7.5 |
| **D — Krylov energy solver** | $\chi_H(t)$ is exactly the numerator matrix of real-time Krylov diagonalisation — no extra circuits | **implemented as a bonus in §8** |
| **E — Your own proposal** | Fourier/LCU filtering, fidelity with product states, ancilla-vs-system consistency checks… | pitch to a mentor by hour 12 |

Three rules apply to every track:

1. **Estimators consume measurement data only.** Exact diagonalisation is for *evaluation*. If
   `exact_spectrum()` appears anywhere inside your estimator, you have cheated.
2. **Uncertainties must be quantified.** A number without an error bar is not a result.
3. **Resources must be reported.** Circuits, shots, depth, gate counts, post-processing time.

---

## 7.1 Track A, step 1 — the honest baseline: windowed DFT + peak ratio

Start with the simplest thing that could work — and hold it to the same rule as everything else:
**the estimator sees only `ts`, `chi` and `chi_q`.** Take a Hann-windowed discrete Fourier
transform of both series, normalised so that an isolated peak's height *is* its spectral weight:

$$\widetilde y(E) \;=\; \frac{\sum_j w_j\, y(t_j)\, e^{+iE t_j}}{\sum_j w_j},$$

then read $\hat q_k = \widetilde{\chi_Q}(E_k)/\widetilde\chi(E_k)$ at each peak.

> **Fourier convention trap.** Because $\chi(t)=\sum_k p_k e^{-iE_kt}$, you must correlate against
> $e^{+iEt}$. Use $e^{-iEt}$ and every recovered energy flips sign — and the symmetry labels will
> still look fine, so the bug is invisible until you compare with theory.

> ## 🎯 Challenge 7 — the honest baseline
>
> Before reaching for anything clever, build the simplest estimator that could work — and hold it to
> the same rule as everything else: **it may see only `ts`, `chi` and `chi_q`.**
>
> **Build** `dft_spectrum(ts, y, e_grid)` and a **data-only** peak finder that locates local maxima
> of $|\tilde\chi(E)|$ and reads $\hat q_k = \tilde\chi_Q(E_k)/\tilde\chi(E_k)$ at each.
>
> ⚠️ **Trap.** Placing your search windows using the known answer is cheating, and it also flatters
> the method: window-edge artefacts get reported as "found peaks" and the baseline looks far better
> than it is. Use `scipy.signal.find_peaks` on the measured spectrum, as here, and let the baseline
> fail where it genuinely fails.
>
> **What to look for in the output.** This baseline is *supposed* to be imperfect. Expect it to miss
> one line entirely, and to land the weakest line's label somewhere near $\hat q \approx 1.2$–$1.5$
> — the right sector, but only just, and with nothing to tell you how close it came to being wrong.
> That number is itself noisy and moves by a few tenths between seeds, so do not tune anything to
> match ours. Both
> failures are diagnosed and fixed in Challenge 8. Record the baseline's numbers now: "we improved
> on the naive method" is a claim you should be able to back with a table.
>
> ⚠️ **Why it misses a line — and why that does not contradict Checkpoint 2.** The bare Fourier
> resolution is $2\pi/T_{\max} \approx 0.50$, comfortably below the $1.03$ line spacing, which is
> exactly what Checkpoint 2 asserts. But the Hann window that suppresses leakage also *widens* every
> peak, to a main lobe of roughly $2\times(2\pi/T_{\max}) \approx 1.0$. The line at
> $E \approx -0.48$ sits $1.03$ from one four times stronger, and is swallowed. This is systematic,
> not shot noise — it happens with noiseless $\chi(t)$ too. Windowing trades resolution for leakage
> suppression, and here the trade costs a whole spectral line.

In [ ]:
def dft_spectrum(ts, y, e_grid):
    """Hann-windowed DFT, normalised so an isolated peak's height ~ its spectral weight."""
    w = np.hanning(len(ts))
    phases = np.exp(1j * np.outer(e_grid, ts))          # note the PLUS sign
    return (phases * (w * y)) @ np.ones(len(ts)) / np.sum(w)


from scipy.signal import find_peaks

E_GRID = np.linspace(-3.2, 3.2, 3200)
F_CHI = dft_spectrum(TS, CHI, E_GRID)
F_Q = dft_spectrum(TS, CHI_Q, E_GRID)


def dft_peak_labels(e_grid, f_chi, f_q, min_height_frac=0.05):
    """Data-only baseline: find local maxima of |chi~(E)|, read the ratio at each.

    NOTE the exact spectrum is used NOWHERE here -- peaks come from the measured data alone,
    which is the rule for every estimator in Part B.
    """
    # ================================================================== TODO
    # Challenge 7: implement dft_peak_labels().
    # Data-only baseline: find local maxima of |chi~(E)| and read the ratio at each.
    #
    # 1. mag = np.abs(f_chi)
    # 2. scipy.signal.find_peaks(mag, height=min_height_frac * mag.max())
    # 3. return a LIST of (e_grid[j], mag[j], Re(f_q[j] / f_chi[j])), ascending in E.
    #    Return a real list, not a generator: the given code calls len() on the result and
    #    then iterates it twice -- once for the table here, again in checkpoint 7.2b.
    #
    # Do NOT use the exact spectrum to place search windows. Peak locations must come from
    # the measured data alone -- that is the rule for every estimator in Part B, and it also
    # stops window-edge artefacts being reported as peaks.
    # ==================================================================
    raise NotImplementedError("Challenge 7: implement dft_peak_labels()")


DFT_PEAKS = dft_peak_labels(E_GRID, F_CHI, F_Q)
print(f"DFT peak-ratio baseline: {len(DFT_PEAKS)} local maxima found in the measured spectrum\n")
print("     found E   peak height    q_ratio  |  nearest exact E   exact q")
dft_labels = {}
for e, h, ratio in DFT_PEAKS:
    k = int(np.argmin(np.abs(E_EXACT - e)))               # evaluation only, not estimation
    dft_labels[k] = ratio
    flag = "" if abs(ratio - Q_EXACT_LABELS[k]) < 0.35 else "   <-- wobbles"
    print(f"   {e:+8.3f}   {h:10.4f}   {ratio:+8.3f}  |  {E_EXACT[k]:+12.3f}   "
          f"{Q_EXACT_LABELS[k]:+6d}{flag}")

_e_dom = E_GRID[int(np.argmax(np.abs(F_CHI)))]
check("DFT sign convention: the dominant line comes out at POSITIVE E", _e_dom > 0,
      f"largest |chi~(E)| at E = {_e_dom:+.3f}; chi(t) = sum_k p_k e^(-iE_k t), so you must "
      f"correlate against e^(+iEt)")
check("the data-only peak finder actually found peaks", len(DFT_PEAKS) >= 2,
      f"{len(DFT_PEAKS)} local maxima")

print("\nTwo separate weaknesses to notice:")
print("  * ENERGIES are poor: the DFT resolution is only 2*pi/T_max ~ 0.5, so peaks are pulled")
print("    toward their neighbours and weak lines can fail to produce a local maximum at all.")
print("  * The weakest LABEL (p ~ 0.05) is contaminated by its neighbours' leakage tails.")
print("Both are fixed by the pencil + shared-frequency least-squares route in the next sections.")

### The picture that explains the whole method

One figure, and if you understand it you understand Track A.

Both curves are windowed Fourier transforms of data taken in the *same* experiment: blue is the
ancilla signal $\chi$, orange is the ancilla-weighted shadow signal $\chi_Q$. They have peaks at
**identical** energies — necessarily, since both are sums of $e^{-iE_kt}$ over the same populated
levels. What differs is the peak *heights*: $\chi$ carries weight $p_k$, $\chi_Q$ carries
$p_k q_k$.

So look at $E \approx +0.55$: the orange curve is three times taller than the blue one, because
that line sits in the $Q=3$ sector. At the $Q=1$ lines the two curves lie on top of each other.
**The ratio of the two curves at a peak is that peak's quantum number.**

Eyeballing only takes you so far, and the figure shows why. At the strong $Q=1$ line near
$E \approx -1.9$ the ratio comes out at $0.97$ — fine. The other two $Q=1$ lines fare worse:
the one near $E \approx -0.48$ produces no local maximum at all and is simply missed, and the
weakest, near $E \approx +1.95$, is dragged to $\approx 1.47$ by leakage from the giant $Q=3$ peak. Deconvolving that leakage is
exactly what §7.2–7.4 add.

In [ ]:
fig, ax = plt.subplots(figsize=(9.4, 3.6))
ax.plot(E_GRID, np.real(F_CHI), color=COL["blue"], lw=1.6, label=r"Re $\tilde\chi(E)$")
ax.plot(E_GRID, np.real(F_Q), color=COL["orange"], lw=1.6, label=r"Re $\tilde\chi_Q(E)$")
for e, q in zip(E_EXACT, Q_EXACT_LABELS):
    ax.axvline(e, color=COL["grid"], lw=1.0, zorder=0)
    ax.annotate(f"q={q:+d}", (e, ax.get_ylim()[1] * 0.86), fontsize=8,
                color=COL["muted"], ha="center")
ax.set_xlabel("energy $E$"); ax.set_ylabel("windowed DFT")
ax.set_title(r"Peak-height ratio $\tilde\chi_Q/\tilde\chi \approx q_k$: "
             "the garbage register labels each line", fontsize=10.5)
ax.legend(frameon=False, fontsize=8.5)
style_axes(ax)
fig.tight_layout()
show(fig, "07_chi_vs_chiQ_spectra")

Look at the $Q=3$ line near $E=+0.55$: the orange curve is **three times taller** than the blue
one, because $\chi_Q$ carries the extra factor $q_k = 3$. At the $Q=1$ lines the two curves sit on
top of each other. That picture *is* the algorithm.

## 7.2 Track A, step 2 — matrix pencil: frequencies without a grid

The DFT resolution is capped at $2\pi/T_{\max}\approx 0.5$, and peak-picking on a grid inherits
that. A **matrix-pencil** (ESPRIT-family) method instead models the signal as a sum of complex
exponentials and solves for the frequencies directly — no grid, resolution limited by noise rather
than by $T_{\max}$.

The recipe for $y_j = \sum_k A_k z_k^{\,j}$ with $z_k = e^{-iE_k\Delta t}$:

1. build the Hankel matrix $Y$ from the samples;
2. split it into $Y_0$ (rows $0..L-1$) and $Y_1$ (rows $1..L$) — a *pencil* $Y_1 - zY_0$;
3. SVD-truncate $Y_0$ to the numerical rank $r$ (this is the denoising step);
4. the eigenvalues of $A = \big(U_r^\dagger Y_1 V_r\big)\Sigma_r^{-1}$ are the $z_k$
   (the code applies $\Sigma_r^{-1}$ on the right — same eigenvalues, since
   $\Sigma^{-1}M$ and $M\Sigma^{-1}$ are similar);
5. $E_k = -\arg(z_k)/\Delta t$.

The singular-value threshold sets $r$, i.e. **how many spectral lines you claim exist**. That is
the one genuinely subjective knob — report it and study its effect.

> ## 🎯 Challenge 8 — matrix-pencil reconstruction
>
> **Build** the estimator that beats the baseline: `matrix_pencil`, `amplitudes_at`, and
> `reconstruct`.
>
> **Spec.**
> `matrix_pencil(y, dt, ...) -> (energies, rank)` extracts the frequencies of
> $y_j = \sum_k A_k e^{-iE_kt_j}$ off-grid, choosing the model order from the singular-value
> spectrum.
> `amplitudes_at(energies, ts, y)` solves the linear least-squares problem for the amplitudes at
> *fixed* frequencies.
> `reconstruct(ts, chi, chi_q, n_modes=None, weight_threshold=0.02)` puts them together, in this
> order: frequencies from $\chi$; amplitudes for $\chi$ at those frequencies; **drop** modes whose
> $|A_k|$ falls below `weight_threshold`; **then** fit $\chi_Q$ on the surviving frequency set;
> sort by energy; return the **4-tuple** `(energies, p_k, q_hat_k, rank)` with
> $p_k = \operatorname{Re}A_k$ and $\hat q_k = \operatorname{Re}(A^Q_k/A_k)$. Challenge 9 needs
> that `rank`.
>
> *(Fitting $\chi_Q$ before or after the drop gives identical numbers whenever nothing is dropped —
> true at the default rank — but they diverge once you force the rank higher, which Going-further
> item 1 asks you to do. Say which you did.)*
>
> **Why the shared-frequency step is the whole trick.** The DFT reads a ratio at a grid point where
> the leakage tails of every neighbouring peak contribute, and on a weak line those tails dominate.
> Fitting all amplitudes simultaneously deconvolves each peak from its neighbours *before* the ratio
> is taken.
>
> ⚠️ **Traps.** (i) Sign: $z_k = e^{-iE_k\Delta t}$, so $E_k = -\arg(z_k)/\Delta t$. Get it backwards
> and your entire spectrum mirrors — and the labels will still look fine, so the labels alone
> will not warn you. Checkpoint 7's items 7.2 and 7.3 will.
> (ii) Matrix pencil **requires a uniform grid**. (iii) The singular-value threshold decides how many
> spectral lines you are claiming exist. It is the one genuinely subjective knob in the method:
> report it, and study its effect.
>
> **You will know it works when** Checkpoint 7 is green.

In [ ]:
def matrix_pencil(y: np.ndarray, dt: float, n_modes: int | None = None,
                  sv_threshold: float = 0.06):
    """Frequencies E_k of  y_j = sum_k A_k exp(-i E_k t_j)  on a uniform grid.

    Returns (energies, rank).  If n_modes is None the rank is chosen as the number of
    singular values above `sv_threshold` times the largest one.
    """
    # ================================================================== TODO
    # Challenge 8: implement matrix_pencil().
    # Frequencies of y_j = sum_k A_k exp(-i E_k t_j) on a uniform grid.
    #
    # Shape of the method (the section above derives it):
    #   * build the Hankel matrix of the samples, scipy.linalg.hankel(y[:ell+1], y[ell:])
    #     with ell = n // 2, and split it into the pencil pair Y0 (rows 0..L-1), Y1 (rows 1..L)
    #   * SVD-truncate Y0 to rank r -- this is the denoising step. With n_modes=None choose r
    #     as the number of singular values above sv_threshold * s[0]
    #   * form the rank-r companion matrix from U_r, Y1, V_r and Sigma_r, and take its
    #     eigenvalues z_k
    #   * convert z_k -> E_k
    #
    # Return (energies, r).
    #
    # Sign check: z_k = exp(-i E_k dt), so E_k = -arg(z_k)/dt. Getting this backwards mirrors
    # your whole spectrum -- and because this benchmark is nearly mirror-symmetric, the labels
    # still round correctly, so 7.4 will not warn you. Neither will 7.2b: the mirrored modes sit
    # CLOSER to the data-only DFT peaks (0.110) than the correct ones (0.149). What catches a
    # sign flip is 7.2 (energies) and 7.3 (weights), both inside Checkpoint 7.
    # ==================================================================
    raise NotImplementedError("Challenge 8: implement matrix_pencil()")


def amplitudes_at(energies: np.ndarray, ts: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Least-squares A_k in  y(t) = sum_k A_k exp(-i E_k t)  at FIXED energies."""
    vand = np.exp(-1j * np.outer(ts, energies))
    return lstsq(vand, y)[0]


def reconstruct(ts, chi, chi_q, n_modes=None, weight_threshold=0.02):
    """Full Track-A pipeline -> (energies, weights p_k, labels q_k, rank), sorted by energy.

    Uses ONLY the measured series.  No exact diagonalisation anywhere.
    """
    # ================================================================== TODO
    # Challenge 8: implement reconstruct().
    # The full Track A pipeline. Uses ONLY ts, chi and chi_q.
    #
    # 1. dt = ts[1] - ts[0];  energies, rank = matrix_pencil(chi, dt, n_modes=n_modes)
    # 2. amps = amplitudes_at(energies, ts, chi)
    # 3. keep only modes with |amps| > weight_threshold
    # 4. amps_q = amplitudes_at(energies, ts, chi_q)   <-- the SAME frequencies, second series
    # 5. sort everything by energy
    # 6. return (energies, Re(amps), Re(amps_q / amps), rank)      <-- a 4-tuple
    #
    # Step 4 is the whole trick: fitting both series on one shared frequency set deconvolves
    # each peak from its neighbours before you take the ratio.
    # ==================================================================
    raise NotImplementedError("Challenge 8: implement reconstruct()")


E_HAT, P_HAT, Q_HAT_LABEL, RANK = reconstruct(TS, CHI, CHI_Q)
print(f"matrix-pencil rank chosen automatically: {RANK}")
print(f"modes surviving the weight threshold   : {len(E_HAT)}\n")
for e, p, q in zip(E_HAT, P_HAT, Q_HAT_LABEL):
    print(f"   E = {e:+8.4f}    p = {p:7.4f}    q_hat = {q:+7.3f}  ->  q = {int(np.rint(q)):+d}")

### Why this beats the DFT ratio

The DFT reads $\widetilde\chi_Q/\widetilde\chi$ at a *grid point*, where the leakage tails of the
three neighbouring peaks all contribute — and on the weak peak the tails dominate. The pencil route
instead (i) locates the frequencies off-grid, then (ii) fits **all** amplitudes simultaneously by
least squares, so each peak's amplitude is deconvolved from the others before the ratio is taken.

## 7.3 Track A, step 3 — uncertainties by parametric bootstrap

Every $\hat\chi(t_j)$ came with a standard error. Resample: perturb each point by its own sem,
re-run the *whole* pipeline, and look at the spread of the recovered $(E_k, p_k, q_k)$. Peaks are
matched to the point estimate by nearest neighbour, with a rejection radius so that a bootstrap
replicate which loses a mode does not corrupt the statistics.

> ## 🎯 Challenge 9 — uncertainties, or it does not count
>
> **Graded by** item **7.5 inside Checkpoint 7 (§7.4)** — four code cells below this one, not
> immediately underneath it. There is no checkpoint directly after this cell; run §7.4 to find
> out whether your bootstrap is right. 7.5 asserts every uncertainty is finite and positive,
> and that the weakest line carries a larger `q` spread than the strongest.
>
> **Build** `bootstrap_uncertainties(...) -> (E_sd, p_sd, q_sd)`: a parametric bootstrap that
> perturbs each measured point by its own standard error, re-runs the **entire** reconstruction
> pipeline, and takes the spread of the recovered quantities.
>
> **Spec.** Match each replicate's peaks to the point estimate by nearest neighbour, with a rejection
> radius so that a replicate which loses a mode contributes `nan` rather than corrupting a different
> peak's statistics. Use `np.nanstd`. Guard the whole thing in `try/except` — an occasional
> replicate will fail to converge, and that must not kill the loop.
>
> **Why re-run the whole pipeline?** Because the rank selection, the frequency extraction and the
> least-squares step all contribute uncertainty. Propagating only the amplitude error would give you
> error bars that are far too small and, worse, confidently wrong on exactly the weak peak where you
> most need them.
>
> **Honest caveat, and a free improvement.** The recipe below perturbs $\chi$ and $\chi_Q$
> **independently**, even though they come from the same shots and their errors are positively
> correlated ($\approx +0.57$ here). That makes the label uncertainties mildly *conservative*.
> Estimating the correlation from your own shot records and resampling the pair jointly is one of
> the cleanest upgrades available — see the Going-further box at the end of §7.4.

In [ ]:
def bootstrap_uncertainties(ts, chi, chi_q, sem, sem_q, n_modes, ref_energies,
                            n_boot=200, seed=0, match_radius=0.35):
    """Parametric bootstrap of the Track-A pipeline. Returns (E_sd, p_sd, q_sd).

    Each is an array of length len(ref_energies), aligned with the point-estimate modes:
    entry k is the spread of the replicate peak matched to ref_energies[k]. The given code
    indexes these with point-estimate indices, so the length must not depend on how many
    peaks an individual replicate happened to find.
    """
    # ================================================================== TODO
    # Challenge 9: implement bootstrap_uncertainties().
    # Parametric bootstrap of the WHOLE reconstruction pipeline.
    #
    # For b in range(n_boot):
    #   * perturb chi   by rng.normal(scale=sem[:, 0]) + 1j*rng.normal(scale=sem[:, 1])
    #   * perturb chi_q the same way with sem_q
    #   * re-run reconstruct(...) with n_modes fixed to the point-estimate rank
    #   * match each recovered peak to ref_energies by nearest neighbour, and only record it
    #     if |e_j - e_ref| < match_radius -- otherwise leave np.nan
    #   * wrap the reconstruct call in try/except: an occasional replicate will not converge
    #     and that must not kill the loop
    #
    # Return (nanstd of E, nanstd of p, nanstd of q) along the replicate axis.
    # Re-running the FULL pipeline matters: rank selection and frequency extraction contribute
    # uncertainty too, and propagating amplitude error alone gives error bars that are far too
    # small exactly on the weak peak where you need them most.
    # ==================================================================
    raise NotImplementedError("Challenge 9: implement bootstrap_uncertainties()")


_t0 = time.time()
E_SD, P_SD, Q_SD = bootstrap_uncertainties(
    TS, CHI, CHI_Q, SWEEP.chi_sem, SWEEP.chi_obs_sem["Q"],
    n_modes=RANK, ref_energies=E_HAT, n_boot=200, seed=sub_seed("bootstrap"))
print(f"200 bootstrap replicates in {time.time() - _t0:.1f} s")

## 7.4 Track A results

Everything below this line is evaluation, not estimation: the reconstruction is already done, and
we are now lining its output up against the exact answer to see how well it did. The table reports,
for each of the four populated levels, the recovered energy, spectral weight and symmetry label
with bootstrap uncertainties beside the exact values.

The line to watch is the last one, at $E \approx +1.95$ with $p \approx 0.05$. It is the hardest by
design, its error bars are the widest, and it is where a better method earns its marks.

In [ ]:
print("Symmetry-resolved spectrum recovered from measurement data only\n")
print(f"{'exact E':>10} {'recovered E':>22} | {'exact p':>9} {'recovered p':>20} |"
      f" {'exact q':>8} {'inferred q':>19}")
print("-" * 100)

ROWS = []
for e0, q0, p0 in zip(E_EXACT, Q_EXACT_LABELS, P_EXACT):
    j = int(np.argmin(np.abs(E_HAT - e0)))
    print(f"{e0:+10.4f} {E_HAT[j]:+13.4f} +- {E_SD[j]:.4f} | {p0:9.4f} "
          f"{P_HAT[j]:11.4f} +- {P_SD[j]:.4f} | {q0:+8d} {Q_HAT_LABEL[j]:+11.3f} +- {Q_SD[j]:.3f}")
    ROWS.append(dict(E_exact=float(e0), E_hat=float(E_HAT[j]), E_sd=float(E_SD[j]),
                     p_exact=float(p0), p_hat=float(P_HAT[j]), p_sd=float(P_SD[j]),
                     q_exact=int(q0), q_hat=float(Q_HAT_LABEL[j]), q_sd=float(Q_SD[j])))

MAX_E_ERR = max(abs(r["E_hat"] - r["E_exact"]) for r in ROWS)
MAX_Q_ERR = max(abs(r["q_hat"] - r["q_exact"]) for r in ROWS)
LABELS_OK = all(int(np.rint(r["q_hat"])) == r["q_exact"] for r in ROWS)
print(f"\nmax |E error| = {MAX_E_ERR:.4f}    max |q error| = {MAX_Q_ERR:.3f}    "
      f"all labels correct after rounding: {LABELS_OK}")

### The Part-B headline figure

The grey shaded curve is the raw ancilla data — the windowed $|\tilde\chi(E)|$ you would get from a
conventional Hadamard test. On its own it is a blurred *three*-lobe envelope: the fourth populated
line, at $E \approx -0.48$, produces no local maximum at all.

The coloured stems are your reconstruction: position = recovered $E_k$, height = recovered $p_k$,
error bars from the bootstrap, **colour = the symmetry sector inferred from the garbage register**,
with the numerical $\hat q_k$ annotated. Black crosses are the exact answer, for evaluation only.

That colour is the deliverable. It is information that simply is not present in the grey curve, and
it cost zero additional circuits.

In [ ]:
SECTOR_COLORS = {3: COL["blue"], 1: COL["orange"], -1: COL["aqua"], -3: COL["violet"]}
e_plot = np.linspace(-3.2, 3.2, 1400)
f_abs = np.abs(dft_spectrum(TS, CHI, e_plot))

fig, ax = plt.subplots(figsize=(9.0, 4.3))
ax.fill_between(e_plot, f_abs, color=COL["grid"], alpha=0.9, lw=0,
                label=r"windowed $|\tilde\chi(E)|$ (ancilla data)")
ax.plot(e_plot, f_abs, color=COL["muted"], lw=1.0)
for e, p, q, esd, psd in zip(E_HAT, P_HAT, Q_HAT_LABEL, E_SD, P_SD):
    c = SECTOR_COLORS.get(int(np.rint(q)), COL["ink"])
    ax.vlines(e, 0, p, color=c, lw=2.4)
    ax.errorbar([e], [p], xerr=[esd], yerr=[psd], fmt="o", color=c, ms=6, lw=1.2, capsize=2)
    ax.annotate(rf"$\hat q={q:+.2f}$", (e, p), textcoords="offset points",
                xytext=(7, 7), fontsize=9, color=c)
ax.plot(E_EXACT, P_EXACT, "x", color=COL["ink"], ms=8, mew=1.7, zorder=5)

from matplotlib.lines import Line2D
handles, _lbls = ax.get_legend_handles_labels()
handles += [Line2D([], [], color=SECTOR_COLORS[3], lw=2.4, label="recovered peak, $Q=3$"),
            Line2D([], [], color=SECTOR_COLORS[1], lw=2.4, label="recovered peak, $Q=1$"),
            Line2D([], [], color=COL["ink"], marker="x", ls="", mew=1.7,
                   label=r"exact $(E_k,\,p_k)$")]
ax.legend(handles=handles, frameon=False, fontsize=8.5, loc="upper left")
ax.set_xlabel("energy $E$"); ax.set_ylabel("spectral weight")
ax.set_title("Symmetry-resolved spectrum from one set of shadow-Hadamard experiments",
             fontsize=11)
style_axes(ax)
fig.tight_layout()
show(fig, "07_symmetry_resolved_spectrum")

### ✅ Checkpoint 7 — Track A acceptance criteria

Eight asserts, mapping one-to-one onto the Track A requirements: at least three energies recovered,
all of them accurate, weights accurate, **every symmetry label correct after rounding**, a
data-only cross-check (7.2b) that the pencil energies line up with the DFT peaks — the one item
that uses no exact reference at all, so it is the piece you can keep in your own pipeline —
uncertainties present for all three quantities, and a demonstrated improvement over the naive DFT
baseline on the weakest line.

Item **7.7** deserves a note, because it is unusual. Rather than *asserting* that the estimator
never peeked at the answer, it tests it: every name that knows the answer — the exact references
*and* the model constructors they could be rebuilt from — is temporarily replaced by a tripwire
object that raises on any attribute access, call or indexing, `reconstruct()` is re-run, and the
result must come out identical.

Be clear about what that does and does not prove. It catches every accidental leak, and every lazy
one. It is **not** a sandbox: an estimator that hard-codes the Hamiltonian's coefficients inline
and diagonalises them would still pass. Treat it as a self-check with teeth — run your own
reconstruction through it, and say so in your report.

In [ ]:
# --------------------------------------------------- CHECKPOINT 7: Track A acceptance criteria
check("7.1  at least three populated energies recovered", len(E_HAT) >= 3, f"{len(E_HAT)} modes", budget_sensitive=True)
check("7.2  all four energies accurate to < 0.05", MAX_E_ERR < 0.05, f"max {MAX_E_ERR:.4f}", budget_sensitive=True)
# A mirrored pencil (E = +arg(z)/dt instead of -arg(z)/dt) still rounds every label correctly on
# this near-symmetric spectrum, so 7.4 cannot catch it. Neither can this one: the mirrored modes
# land 0.110 from the DFT peaks against the correct pencil's 0.149, so 7.2b is if anything
# happier with the wrong answer. A sign flip is caught by 7.2 (energies, 0.074 > 0.05) and 7.3
# (weights, 0.094 > 0.02) -- both of which consult the exact reference. What 7.2b is good for is
# that it uses only measured data, so it is the one item here you can keep in your own pipeline.
_dft_e = np.array([e for e, _h, _r in DFT_PEAKS])
_matched = [np.min(np.abs(E_HAT - e)) for e in _dft_e]
check("7.2b pencil energies are consistent with the data-only DFT peaks",
      max(_matched) < 0.3,
      f"worst DFT peak-to-pencil distance {max(_matched):.3f} "
      f"(this is a data-only consistency check: it catches a pencil that has wandered off the "
      f"measured spectrum, but NOT a global sign flip -- see 7.2 and 7.3 for that)",
      budget_sensitive=True)

check("7.3  spectral weights accurate to < 0.02",
      max(abs(r["p_hat"] - r["p_exact"]) for r in ROWS) < 0.02,
      f"max {max(abs(r['p_hat'] - r['p_exact']) for r in ROWS):.4f}", budget_sensitive=True)
check("7.4  every symmetry label correct after rounding", LABELS_OK,
      "  ".join(f"{r['q_hat']:+.2f}->{int(np.rint(r['q_hat'])):+d}" for r in ROWS), budget_sensitive=True)
check("7.5  uncertainties reported for every quantity, and they behave sensibly",
      np.all(np.isfinite(E_SD)) and np.all(np.isfinite(P_SD)) and np.all(np.isfinite(Q_SD))
      and np.all(E_SD > 0) and np.all(P_SD > 0) and np.all(Q_SD > 0)
      and Q_SD[int(np.argmin(P_HAT))] > 2 * Q_SD[int(np.argmax(P_HAT))],
      f"weakest-line q sd {Q_SD[int(np.argmin(P_HAT))]:.3f} vs strongest "
      f"{Q_SD[int(np.argmax(P_HAT))]:.3f} -- the weak peak MUST be the uncertain one",
      budget_sensitive=True)
_weak = int(np.argmin(P_EXACT))                       # the p ~ 0.05 line
_pencil_weak = Q_HAT_LABEL[int(np.argmin(np.abs(E_HAT - E_EXACT[_weak])))]
if _weak in dft_labels:
    # The DFT ratio on the weak line is itself a noisy random variable (median ~1.4, 5th pct ~1.2),
    # so a bare head-to-head fails ~6% of correct runs. Pass if the pencil wins OR is simply good.
    _beats = abs(_pencil_weak - Q_EXACT_LABELS[_weak]) < abs(dft_labels[_weak] - Q_EXACT_LABELS[_weak])
    check("7.6  pencil label on the weakest peak beats the DFT ratio, or is accurate anyway",
          _beats or abs(_pencil_weak - Q_EXACT_LABELS[_weak]) < 0.35,
          f"pencil {_pencil_weak:+.3f} vs DFT {dft_labels[_weak]:+.3f} "
          f"(exact {Q_EXACT_LABELS[_weak]:+d})"
          + ("" if _beats else "  [pencil lost the head-to-head but is within 0.35 -- the DFT"
                               " ratio is noisy, this happens]"),
          budget_sensitive=True)
else:
    check("7.6  pencil recovers a peak the naive DFT missed entirely", True,
          f"pencil {_pencil_weak:+.3f} at E = {E_EXACT[_weak]:+.3f}; no DFT local maximum there", budget_sensitive=True)


# 7.7 is a real tripwire, not a tautology: re-run the estimator with every name that KNOWS THE
# ANSWER -- the exact references AND the model constructors they could be rebuilt from -- replaced
# by an object that raises on any use, and demand an identical result.
class _Tripwire:
    def __init__(self, name):
        self._n = name
    def __getattr__(self, k):
        raise AssertionError(f"estimator touched the exact reference `{self._n}`")
    def __call__(self, *a, **k):
        raise AssertionError(f"estimator called the exact reference `{self._n}`")
    def __getitem__(self, k):
        raise AssertionError(f"estimator indexed the exact reference `{self._n}`")


_HIDE = [  # exact references ...
    "exact_spectrum", "exact_chi", "exact_chi_O", "exact_unitary", "exact_state",
    "exact_system_marginal_expectation", "SPEC", "E_EXACT", "P_EXACT", "Q_EXACT_LABELS",
    # ... and the model itself, or an estimator could simply rebuild and diagonalise it
    "HAM", "CHARGE", "PSI", "Z0_OBS",
    "benchmark_hamiltonian", "conserved_charge", "state_prep_circuit",
    "SweepResult", "SWEEP", "CHI_REF", "CHI_Q_REF",
]
_HIDE = [k for k in _HIDE if k in globals()]
_saved = {k: globals()[k] for k in _HIDE}
try:
    globals().update({k: _Tripwire(k) for k in _HIDE})
    _blind = reconstruct(TS, CHI, CHI_Q)                 # measurement data only
finally:
    globals().update(_saved)

check("7.7  the estimator does not reach for anything that knows the answer",
      np.allclose(_blind[0], E_HAT) and np.allclose(_blind[1], P_HAT)
      and np.allclose(_blind[2], Q_HAT_LABEL),
      f"reconstruct() reproduced its answer with {len(_HIDE)} names replaced by tripwires")

> ### 🧪 Going further — Track A, where the marks are
> 1. **Rank selection.** `sv_threshold=0.06` is a magic number. Sweep it, plot the singular-value
>    spectrum, and adopt a principled criterion (MDL, AIC, a noise-floor estimate from the sem).
>    Show what happens when you over- and under-estimate the rank.
> 2. **Correlated bootstrap.** Estimate $\operatorname{corr}(\hat\chi,\hat\chi_Q)$ per time point
>    from the shot records themselves (you have them!), then resample the pair jointly. Report
>    how much the label error bars shrink.
> 3. **Alternative estimators.** Non-negative least squares or LASSO on a dense energy grid;
>    Prony; ESPRIT with forward-backward averaging; a Bayesian fit with a positivity prior on
>    $p_k$. Which is most robust at 4× fewer shots?
> 4. **Push until it breaks.** Halve the shots, halve $T_{\max}$, or add a fifth level that is
>    nearly degenerate with an existing one. Find and *report* the failure boundary — honest
>    failure analysis is an explicit rubric line.
> 5. **Sector projectors.** Instead of labelling peaks, estimate the projector
>    $\Pi_q$ and reconstruct $\chi_q(t)=\operatorname{Tr}[\Pi_q U(t)\rho]$ directly. $\Pi_q$ is a
>    polynomial in $Q$, so it is a linear combination of Pauli strings — the same records work.
> 6. **Adaptive time grids.** You are free to re-space $t_j$ under the same $T_{\max}$ and total
>    shots. Does a non-uniform grid help? (Careful: matrix pencil *requires* uniform sampling —
>    what would you use instead?)

---

## 7.5 Scaffolds for Tracks B, C and E

These are **not implemented** — they are your alternative advanced goals. Each one is reachable
from the machinery already in this notebook; the code sketches below show where to start. Agree
concrete acceptance criteria with a mentor by the hour-12 checkpoint if you take one.

### Track B — dynamics A/B tester (anti-controlled Hadamard test)

Replace the controlled-$U$ with "$W$ when the ancilla is $|0\rangle$, $U$ when it is $|1\rangle$".
The ancilla then measures the **overlap of two dynamics**, $\operatorname{Tr}[W^\dagger U\rho]$,
while the shadows give you the average ($\rho^{(I)}\propto W\rho W^\dagger + U\rho U^\dagger$) and
the interference. Their *difference* needs an $X$-basis ancilla readout — concretely, **delete the
final Hadamard** (paper Appendices B and E). Natural study on this benchmark: exact vs. Trotterised
evolution, or first- vs. second-order product formulas, or two transpiler settings. Deliver
overlap-vs-time curves, per-observable disagreement profiles, and a gate-cost comparison.

```python
def build_ab_circuit(ham_u, ham_w, t, phi, basis, final_hadamard=True):
    sys_reg, anc_reg = QuantumRegister(N_SYS, "sys"), QuantumRegister(1, "anc")
    qc = QuantumCircuit(sys_reg, anc_reg)
    qc.compose(state_prep_circuit(), qubits=sys_reg, inplace=True)
    qc.h(anc_reg[0])
    qc.x(anc_reg[0])                                                  # anti-control W ...
    qc.append(build_controlled_evolution(ham_w, t, "exact"), [anc_reg[0], *sys_reg])
    qc.x(anc_reg[0])                                                  # ... undo
    qc.append(build_controlled_evolution(ham_u, t, "exact"), [anc_reg[0], *sys_reg])
    if phi != 0.0:
        qc.p(phi, anc_reg[0])
    if final_hadamard:
        qc.h(anc_reg[0])        # Z-basis ancilla readout -> rho^(Z)
    # else: X-basis readout -> rho^(X) = (W rho W+ - U rho U+)/2
    ...
```

Budget note: this roughly doubles the controlled-evolution depth.

### Track C — eigenstate detective

For a true eigenstate, $|\chi(t)| \equiv 1$ with a single rotating phase, and *every* unweighted
shadow observable of $\rho^{(I)}(t)$ is time-independent. Superpositions betray themselves twice
over. Build witnesses from both facts, calibrate on a family interpolating between the exact
eigenstate $|000\rangle$ (use `state_prep_circuit` with `ry(θ, 0)`, $\theta\to0$) and strongly
mixed-sector states, and report detection power vs. shot budget.

```python
def eigenstateness_witnesses(sweep):
    w1 = 1.0 - np.mean(np.abs(sweep.chi))                  # magnitude deficit
    o = SparsePauliOp.from_sparse_list([("Z", [0], 1.0)], N_SYS)
    traj = np.array([estimate_system_observable(list(p), o)[0] for p in sweep.records])
    w2 = float(np.std(traj))                               # drift of a rho^(I) observable
    return w1, w2
```

Stretch goal (feasible at $n=3$, exponentially costly in general): estimate
$\operatorname{Tr}[(\rho^{(I)})^2]$ from pairs of shadow snapshots and use
$\operatorname{Tr}[(\rho^{(I)})^2]=1 \iff \rho$ is a pure eigenstate of $U$.

> **Before you spend a day on it:** for a *pure* input state,
> $\operatorname{Tr}[(\rho^{(I)})^2] = \tfrac12\big(1+|\chi(t)|^2\big)$ exactly, so on an ideal
> simulator the purity witness carries **no information beyond $|\chi(t)|$** — which witness $w_1$
> already uses, far more cheaply. Its real value is as a *mixedness* diagnostic: under noise the
> identity breaks, and the gap between the measured purity and $(1+|\chi|^2)/2$ is a decoherence
> witness. Frame it that way and it becomes a good result rather than an expensive tautology.

### Track E — your own proposal

Raw material in Ref. [1] that nobody has used yet: Fourier-filter / linear-combination-of-unitaries
post-processing of the same time series (statistical phase estimation); fidelity with product
states from local shadows (the projector onto $|000\rangle$ expands into 8 Pauli strings, all of
which you can already estimate); ancilla-versus-system consistency cross-checks in the spirit of
verified phase estimation. Bring a measurable success criterion and a shot budget.

---

# 8 · PART C bonus (Track D) — a Krylov energy solver from the *same* records

Here is the observation that makes this a bonus rather than a new experiment. Real-time Krylov
diagonalisation builds the subspace $\{U(t_j)|\psi\rangle\}_{j<m}$ and solves the generalised
eigenproblem $\mathsf H c = E\,S\,c$ with

$$S_{jk} = \langle\psi|U(t_k-t_j)|\psi\rangle = \chi(t_k-t_j),
\qquad
\mathsf H_{jk} = \langle\psi|H\,U(t_k-t_j)|\psi\rangle = \chi_H(t_k-t_j).$$

Ordinarily $\mathsf H$ would need a *separate* modified Hadamard test per Pauli term of $H$. Here
$\chi_H(t)$ is just another ancilla-weighted shadow average — we already computed it in §6.1. Zero
new circuits. We use $\chi(-t)=\chi(t)^*$ to fill in negative lags.

> **This is a research-grade problem, not a formula.** The noisy GEVP is ill-conditioned: too large
> $m$ or too small a threshold and shot noise pushes the estimate **below** the true value. Also
> note Krylov only sees sectors the input state overlaps — the global ground level of this
> benchmark ($E=-2.635$, in $Q=-1$) is **invisible** to these records. The target is the lowest
> *populated* level, $-1.9152$.

> ## 🎯 Challenge 10 *(bonus)* — a Krylov energy solver, for free
>
> **Graded advisorily.** The only check on this one is `soft=True`: it reports but never fails,
> because the GEVP is genuinely stochastic run to run. A green notebook therefore tells you
> nothing here — judge it yourself by whether `E0` lands near the lowest *populated* level
> (≈ −1.92), not the global ground state (−2.635).
>
> **Build** `krylov_lowest_energy(ts, chi, chi_h, m=16, threshold=2e-2) -> (float E0, int n_kept)`:
> assemble
> $S_{jk} = \chi(t_k - t_j)$ and $\mathsf H_{jk} = \chi_H(t_k - t_j)$ from the series you already
> have, symmetrise them, and solve the thresholded generalised eigenproblem
> $\mathsf H c = E\,S\,c$.
>
> **Spec.** Use $\chi(-t) = \chi(t)^*$ for negative lags. Require a uniform grid starting at $t=0$
> and raise if given anything else. Regularise by canonical orthogonalisation: diagonalise $S$, keep
> eigenvalues above `threshold * max`, and project $\mathsf H$ into that subspace.
>
> **Why this is a bonus and not a whole track's work:** ordinarily the numerator matrix would need a
> separate modified Hadamard test *per Pauli term of $H$*. Here $\chi_H(t)$ is just another
> ancilla-weighted shadow average, already computed in §6.1. **Zero new circuits.**
>
> ⚠️ **Two traps, both instructive.** (i) The noisy GEVP is ill-conditioned — under-regularise and
> the estimate dives *below* the true energy, because it is no longer variational once $S$ is noisy.
> (ii) Krylov only reaches sectors the input state overlaps: the global ground level of this
> benchmark ($E = -2.635$, in $Q=-1$) is **invisible** to these records. Your target is the lowest
> *populated* level, $-1.9152$. Claiming you found the ground state would be wrong.

In [ ]:
def krylov_lowest_energy(ts, chi, chi_h, m=16, threshold=2e-2):
    """Thresholded real-time Krylov GEVP from measured series only.

    Requires a uniform grid starting at t = 0. Uses chi(-t) = conj(chi(t)).
    """
    # ================================================================== TODO
    # Challenge 10: implement krylov_lowest_energy().
    # Real-time Krylov GEVP from the measured series only.
    #
    # 1. guard: uniform grid starting at t=0, and m <= len(ts)
    # 2. series(y, k) = y[k] if k >= 0 else conj(y[-k])       (chi(-t) = conj(chi(t)))
    # 3. S[j,k] = series(chi, k-j);  H[j,k] = series(chi_h, k-j);  Hermitise both
    # 4. eigh(S), keep eigenvalues > threshold * max, proj = evecs[:, keep]/sqrt(evals[keep])
    # 5. H_red = proj^dag H proj;  return (float(smallest eigenvalue of H_red via
    #    np.linalg.eigvalsh), int(number kept)). Use eigvalsh, not eigvals: the latter hands
    #    back a complex scalar that passes every checkpoint and then breaks json.dump in the
    #    final cell with 'Object of type complex128 is not JSON serializable'.
    #
    # Step 4 is canonical orthogonalisation and it is the only thing standing between you and
    # a wildly negative answer. Under-regularise and the estimate dives below the true energy.
    # ==================================================================
    raise NotImplementedError("Challenge 10: implement krylov_lowest_energy()")


E0_EXACT_POPULATED = float(E_EXACT[0])
E0_KRYLOV, KEPT = krylov_lowest_energy(TS, CHI, CHI_H, m=min(16, N_TIMES), threshold=2e-2)
print(f"Krylov GEVP (m = {min(16, N_TIMES)}, threshold = 0.02, {KEPT} vectors kept)")
print(f"  estimate          {E0_KRYLOV:+.4f}")
print(f"  lowest populated  {E0_EXACT_POPULATED:+.4f}   (error {abs(E0_KRYLOV - E0_EXACT_POPULATED):.4f})")
print(f"  global ground     {float(np.min(SPEC.energies)):+.4f}   <-- unpopulated, invisible here")
print(f"  matrix-pencil     {float(E_HAT[0]):+.4f}   (the §7 pipeline, for comparison)")

### Watching the regularisation fail

The single number above is not the interesting part; this figure is. It sweeps the eigenvalue
threshold over three decades for four subspace dimensions, against two reference lines: the true
lowest *populated* level (dashed) and the true global ground state (dotted, unpopulated and
therefore unreachable).

Three things to read off it:

1. **Too small a threshold and the estimate dives**, in places far below the global ground state.
   The GEVP is not variational once $S$ is noisy — the whole point of the bonus.
2. **Too large a threshold and it rises**, because you have thrown away genuine Krylov directions.
3. **A usable band exists only at the right $m$.** At $m=6$ and $m=10$ the curve plateaus broadly
   but $0.2$–$0.6$ *above* the truth: the subspace is too small to contain the state. At $m=24$ it
   never comes within $0.15$ at any threshold. Only $m=16$ has a band — roughly $0.007$ to $0.1$ —
   that lands on the answer. Locating that operating point *without* knowing the answer is the
   actual research problem.

The checkpoint below is deliberately **advisory**. With this shot budget the estimate lands within
$0.15$ of the truth only about half the time, and sits *below* it more often than not (≈ 55 % of
parametric-bootstrap replicates). Do not
"fix" that by tightening the tolerance; fix it with better regularisation, and *measure* how often
you violate the bound.

In [ ]:
# --------- the ill-conditioning, made visible: sweep the threshold and the subspace dimension
thresholds = np.logspace(-4, -0.7, 26)
dims = [6, 10, 16, 24] if N_TIMES >= 24 else [4, 6, 10, min(16, N_TIMES)]

fig, ax = plt.subplots(figsize=(9.0, 3.9))
for m, c in zip(dims, [COL["blue"], COL["orange"], COL["aqua"], COL["violet"]]):
    vals = [krylov_lowest_energy(TS, CHI, CHI_H, m=m, threshold=th)[0] for th in thresholds]
    ax.semilogx(thresholds, vals, "o-", ms=3.2, lw=1.4, color=c, label=f"m = {m}")
ax.axhline(E0_EXACT_POPULATED, color=COL["ink"], lw=1.4, ls="--",
           label="exact lowest populated")
ax.axhline(float(np.min(SPEC.energies)), color=COL["muted"], lw=1.0, ls=":",
           label="global ground (unpopulated)")
ax.set_ylim(min(-3.4, float(np.min(SPEC.energies)) - 0.8), 0.4)   # else one wild point flattens it
ax.set_xlabel("eigenvalue threshold on $S$")
ax.set_ylabel(r"$\hat E_0$")
ax.set_title("Krylov GEVP: under-regularise and shot noise pushes you below the truth",
             fontsize=10.5)
ax.legend(frameon=False, fontsize=8, ncol=2)
style_axes(ax)
fig.tight_layout()
show(fig, "08_krylov_regularisation")

# ADVISORY, deliberately: with the measured shot noise this estimator lands within 0.15 of the
# truth only about half the time at m=16, threshold=0.02, and sits BELOW it more often than not
# (~55% of parametric-bootstrap replicates).
# That instability IS the bonus problem -- do not "fix" it by tightening the tolerance.
check("Krylov targets the lowest POPULATED level, not the global ground state",
      abs(E0_KRYLOV - E0_EXACT_POPULATED) < 0.6
      and abs(E0_KRYLOV - float(np.min(SPEC.energies))) > 0.3,
      f"E0_krylov = {E0_KRYLOV:+.4f}", soft=True)

> ### 🧪 Going further — Track D
> 1. Replace the eigenvalue cut with a principled regulariser: Tikhonov on $S$, a noise-floor cut
>    derived from the measured sems, or the "fast Krylov" thresholding of Klymko *et al.*
> 2. Quantify the **variational violation rate**: over many bootstrap replicates, how often does
>    $\hat E_0$ fall below the true lowest populated energy? Plot that rate against $m$ and the
>    threshold; pick the operating point that keeps it under, say, 5 %.
> 3. Compare Krylov against the §7 matrix-pencil estimate at equal shot budget. Which is more
>    accurate, and *why*?
> 4. Use both $\chi$ and $\chi_H$ **and** $\chi_Q$ to run Krylov **inside a single symmetry
>    sector** — a symmetry-resolved ground-state estimate. (Hint: $\Pi_q$ is a polynomial in $Q$.)

---

# 9 · PART C bonus — noise robustness

A quick, honest look at what happens on a noisy device. We use a generic depolarising + readout
error model and the **Trotter** circuit path (the exact `UnitaryGate` is not hardware-native, so
noising it is not meaningful).

What to expect: readout error biases *both* the ancilla and the shadow snapshots, and gate noise
damps $|\chi(t)|$ roughly exponentially in circuit depth — which in a spectral reconstruction looks
like **artificial line broadening**. Claims here are limited to robustness. **No advantage claims.**

> ## 🎯 Challenge 11 *(bonus)* — how much of this survives noise? *(given complete — read it, do not rewrite it)*
>
> **Read** the noise model and the *same* sweep run through it. Two sweeps, identical seeds and
> identical circuits, differing **only** in the backend — which is the controlled comparison you
> need in order to attribute any difference to noise rather than to statistics.
>
> **How it is built, and why.** Depolarising error on one- and two-qubit gates plus a readout
> confusion matrix is enough to make the point. It uses the **Trotter** path deliberately: the exact
> `UnitaryGate` is not hardware-native, so attaching gate noise to it would not mean anything
> physical. It sub-samples the time grid rather than truncating it, so $T_{\max}$ — and therefore
> the spectral resolution — is preserved.
>
> **Rules of engagement.** Claims here are limited to **robustness**. No advantage claims, no
> "quantum supremacy of the garbage register". The valuable output is an honest ranking of which of
> your Part-B numbers survive and which do not — and the answer is more interesting than you might
> expect, because $\hat q_k$ is a ratio taken on the same shots.

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

def simple_noise_model(p1=3e-4, p2=6e-3, p_ro=1.2e-2) -> NoiseModel:
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(p1, 1), ["rz", "sx", "x", "h", "s", "sdg"])
    nm.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ["cx"])
    nm.add_all_qubit_readout_error(ReadoutError([[1 - p_ro, p_ro], [p_ro, 1 - p_ro]]))
    return nm


# Sub-sample the grid rather than truncating it: this keeps the full T_max (so the spectrum
# stays resolvable) while cutting the number of noisy Trotter circuits by 3x.
STRIDE = 3
TS_N = TS[::STRIDE]
N_NOISE = len(TS_N)
SHOTS_NOISE = min(SHOTS, 800)

print(f"two sweeps: {N_NOISE} times (stride {STRIDE}, T_max {TS_N[-1]:.1f}) "
      f"x 2 quadratures x {SHOTS_NOISE} shots, trotter reps=1 ...")
_t0 = time.time()
# identical seeds and identical circuits; the ONLY difference is the backend
sweep_ideal_tr = run_time_sweep(HAM, TS_N, SHOTS_NOISE, sub_seed("noise-sweep"),
                                joint_observables={"Q": CHARGE}, method="trotter", reps=1,
                                backend=AerSimulator(), verbose=False)
sweep_noisy = run_time_sweep(HAM, TS_N, SHOTS_NOISE, sub_seed("noise-sweep"),
                             joint_observables={"Q": CHARGE}, method="trotter", reps=1,
                             backend=AerSimulator(noise_model=simple_noise_model()), verbose=False)
chi_noisy = sweep_noisy.chi
print(f"done in {time.time() - _t0:.1f} s")

### Reading the damage honestly

Two panels, one message each.

**Left** — the time-domain signal. Noise multiplies $\chi(t)$ by roughly a constant factor:
depolarising error at fixed circuit depth acts close to a uniform contraction, so the curve keeps
its shape and loses amplitude. Measured here, the per-time damping ratio has mean $0.58$ with a
standard deviation of only $0.04$, and the induced phase shift is $0.05$ rad — a contraction, not a
distortion.

**Right** — the same thing in the spectral domain, where it becomes the statement that matters for
Track A: peak **positions** survive, peak **weights** do not. A damped $\chi$ still oscillates at
the same frequencies, so your recovered $E_k$ are largely safe; your $p_k$ are biased low by
roughly the damping factor printed underneath.

Now ask the question the Going-further box asks: what happens to $\hat q_k$? It is a *ratio* of two
quantities measured on the same shots, so a common multiplicative damping should cancel. If you can
show that quantitatively, you have the most interesting robustness result available in this
challenge — and you already have the data (`sweep_noisy.chi_obs["Q"]`).

In [ ]:
chi_ref_n = exact_chi(HAM, PSI, TS_N)

fig, axes = plt.subplots(1, 2, figsize=(10.6, 3.6))
ax = axes[0]
ax.plot(TS_N, chi_ref_n.real, color=COL["muted"], lw=1.5, label="exact")
ax.plot(TS_N, sweep_ideal_tr.chi.real, "o-", ms=3.2, lw=1, color=COL["blue"],
        label="ideal, Trotter reps=1")
ax.plot(TS_N, chi_noisy.real, "s-", ms=3.2, lw=1, color=COL["orange"], label="noisy")
ax.set_xlabel("t"); ax.set_ylabel(r"Re $\chi(t)$")
ax.set_title("Noise damps the signal", fontsize=10)
ax.legend(frameon=False, fontsize=8)

ax = axes[1]
eg = np.linspace(-3.2, 3.2, 1200)
ax.plot(eg, np.abs(dft_spectrum(TS_N, chi_ref_n, eg)), color=COL["muted"], lw=1.5, label="exact")
ax.plot(eg, np.abs(dft_spectrum(TS_N, sweep_ideal_tr.chi, eg)), lw=1.5, color=COL["blue"],
        label="ideal")
ax.plot(eg, np.abs(dft_spectrum(TS_N, chi_noisy, eg)), lw=1.5, color=COL["orange"], label="noisy")
ax.set_xlim(-3.2, 3.2)
for e in E_EXACT:
    ax.axvline(e, color=COL["grid"], lw=1.0, zorder=0)
ax.set_xlabel("energy $E$"); ax.set_ylabel(r"$|\tilde\chi(E)|$")
ax.set_title("...which shows up as loss of spectral weight", fontsize=10)
ax.legend(frameon=False, fontsize=8)

for a in axes:
    style_axes(a)
fig.tight_layout()
show(fig, "09_noise_comparison")

damp = float(np.mean(np.abs(chi_noisy)) / np.mean(np.abs(sweep_ideal_tr.chi)))
print(f"mean |chi| ratio  noisy / ideal = {damp:.3f}")
check("the noise model actually does something", damp < 0.9,
      f"noisy/ideal mean |chi| = {damp:.3f}", soft=True)
print("The peak POSITIONS survive; the WEIGHTS do not. Which of your Part-B numbers is")
print("robust and which is not? Answering that honestly is worth more than a mitigation hack.")

> ### 🧪 Going further — robustness
> 1. **Readout mitigation** on the ancilla only vs. on all four qubits — how much of the weight
>    loss comes back? (`mthree`, or a 2×2 confusion-matrix inversion you write yourself.)
> 2. **Does the symmetry label survive?** $\hat q_k$ is a *ratio* of two quantities measured on the
>    same shots, so depolarising damping should largely cancel. Test that claim — it is the most
>    interesting robustness statement available in this challenge. You already have everything you
>    need: `sweep_noisy.chi` and `sweep_noisy.chi_obs["Q"]` go straight into `reconstruct()`.
> 3. **Sector post-selection as error detection.** $Q$ is conserved, so a shot whose $Z$-basis
>    snapshot implies a forbidden charge is a detected error. Quantify the yield/accuracy trade-off
>    (careful: post-selection only applies to shots that happened to draw the all-$Z$ basis).
> 4. Run a reduced two-qubit instance on real hardware if you have access. Robustness claims only.

---

# 10 · Wrap-up

The cell below does three things, and the third is the one that matters for your submission.

First it **replays every checkpoint** in order and prints a one-line verdict for each, so you get a
single scrollable audit of the whole notebook rather than having to hunt back through fifty cells.
Second it **fails loudly** if any *hard* checkpoint did not pass — advisory warnings are listed
separately and do not abort, because two of them (the Krylov GEVP, and everything at a reduced
budget) are stochastic by design and a red notebook for those reasons would be misleading.

Third, it writes **`run_summary.json`**: budget, seed, package versions, circuit and shot counts,
wall-clock time, every headline number from Parts A and B, the full Track A results table, and the
checkpoint tally. That file is what makes your run *auditable* — a judge can diff it against their
own re-run without reading a single plot. Attach it together with the `./figures/` directory (eight
PNGs, written as each figure was drawn) and your code, and most of the §10.1 checklist below is
already done.

In [ ]:
# ------------------------------------------------------------------ full checkpoint summary
passed = sum(1 for _, ok, _, _ in CHECK_LOG if ok)
hard_failed = [n for n, ok, _, soft in CHECK_LOG if not ok and not soft]
soft_failed = [n for n, ok, _, soft in CHECK_LOG if not ok and soft]
print(f"{passed} / {len(CHECK_LOG)} checkpoints passed"
      f"   ({len(hard_failed)} hard failures, {len(soft_failed)} advisory warnings)\n")
for name, ok, detail, soft in CHECK_LOG:
    print(f"  {'ok  ' if ok else ('warn' if soft else 'FAIL')}  {name}")
assert not hard_failed, f"hard checkpoint failures: {hard_failed}"

SUMMARY = dict(
    budget=BUDGET, seed=SEED, n_times=N_TIMES, dt=DT, shots=SHOTS,
    qiskit=qiskit.__version__, qiskit_aer=qiskit_aer.__version__,
    total_circuits=SWEEP.total_circuits, total_shots=SWEEP.total_shots,
    sweep_wall_seconds=round(WALL, 2),
    chi_rms_error=float(np.sqrt(np.mean(err ** 2))),
    chi_mean_predicted_sem=float(np.mean(sem)),
    chi_max_err_over_sem=float(np.max(err / sem)),
    chi_q_rms_error=float(np.sqrt(np.mean(err_q ** 2))),
    H_hat=H_HAT, H_sem=H_SEM, H_exact=H_EXPECT_EXACT,
    Q_hat=Q_HAT, Q_sem=Q_SEM, Q_exact=Q_EXPECT_EXACT,
    Z0_t=float(TS[PROBE_IDX]), Z0_hat=Z0_HAT, Z0_sem=Z0_SEM, Z0_exact=Z0_REF,
    scaling_shots=list(SCALE_NS), scaling_mean_abs_err=scale_err, scaling_slope=SCALE_SLOPE,
    pencil_rank=int(RANK), rows=ROWS,
    max_E_error=MAX_E_ERR, max_q_error=MAX_Q_ERR, labels_all_correct=bool(LABELS_OK),
    krylov_E0=E0_KRYLOV, krylov_E0_exact=E0_EXACT_POPULATED,
    checkpoints_passed=passed, checkpoints_total=len(CHECK_LOG),
    checkpoints_advisory_warnings=soft_failed,
)
with open("run_summary.json", "w") as fh:
    json.dump(SUMMARY, fh, indent=2)
print("\nwrote run_summary.json  (attach this to your submission)")
print(f"figures saved to ./{FIGDIR}/: " + ", ".join(sorted(os.listdir(FIGDIR))))
print("on Colab, download them with:  from google.colab import files; files.download('run_summary.json')")

## 10.1 What to submit

* a **reproducible repository** with an environment file and frozen seeds;
* this notebook (or your own), rerun clean from top to bottom;
* **circuit diagrams** for both quadratures;
* validation plots for $\chi(t)$, $\langle H\rangle$ and one more observable;
* the **error-vs-shots** plot;
* your track's headline result **with uncertainties** — Track A: the reconstructed spectrum with
  weights and symmetry labels;
* a **resource table**: circuits, total shots, transpiled 1q/2q gate counts, depth, classical
  post-processing time;
* an honest **failure-case discussion** (Track A: unresolved peaks, weak weights, near
  degeneracies);
* a five-minute demo.

(Every figure in this notebook is written to `./figures/` as it is drawn, and the headline
numbers land in `run_summary.json` — both are ready to attach.)

| Category | Weight | Evidence |
|---|---:|---|
| Fundamental correctness | 30 % | correct circuits, unbiased estimators, exact-simulation agreement |
| Advanced application (chosen track) | 25 % | recovery and insight quality, uncertainty analysis, honest failure cases |
| Resource efficiency | 20 % | accuracy under the fixed shot/depth budget |
| Robustness | 10 % | noise analysis, mitigation, stable resampling |
| Originality | 10 % | adaptive measurements, better reconstruction, new applications |
| Reproducibility & communication | 5 % | clean repository, report, demo |

## 10.2 Suggested 48-hour schedule

| Hours | Focus |
|---|---|
| 0–6 | run this notebook end to end; read §1 and §3.4; inspect the circuits |
| 6–18 | fundamental goal: re-derive the estimators yourself, extend the validation, shot-scaling study. **Commit to an advanced track by hour 12.** |
| 18–34 | your chosen track (Track A: reconstruction, labels, bootstrap) |
| 34–44 | optimisation or a bonus layer; resource metrics |
| 44–48 | clean rerun, freeze seeds, figures, report, presentation |

## 10.3 Pitfalls checklist

1. **Imaginary-quadrature sign.** Pin $\phi=-\pi/2 \Rightarrow \operatorname{Im}\chi$ with the
   statevector test in §3.3 before trusting any spectrum. Do **not** debug against Appendix A of
   the **arXiv version** of
   Ref. [1] — see §1.1.
2. **Two different endiannesses.** Pauli labels (rightmost = qubit 0) and memory bitstrings
   (leftmost = highest classical bit). §2 rule #1 and §3.2 rule #2.
3. **`control()` before or after synthesis?** Know which object you built (§3.1).
4. **Pooling rules.** Unweighted observables that commute with the *implemented* evolution pool
   over everything; $\chi_O(t)$ is per-$t$ *and* per-quadrature. Pooling $\chi_O$ across $t$ is a
   bug, not a bonus.
   **On the Trotter path $H$ no longer commutes with the implemented $U$** — $Q$ still does — so
   pooling $\langle H\rangle$ over times buys statistics at the price of a Trotter bias
   ($\approx-0.02$, i.e. $\approx2.6\sigma$, at `reps=1` on this benchmark). See §1.2.
5. **The identity Pauli term.** $\hat P\equiv1$, so its ancilla-weighted average is just $\chi(t)$.
   Handle observables with an identity component accordingly.
6. **Weight-2 shadow terms** have single-shot second moment $3^2=9$, roughly $3\times$ the variance
   of weight-1 terms. $\langle H\rangle$ converges more slowly than $\langle Q\rangle$.
7. **Fourier conventions.** $\chi(t)=\sum_k p_k e^{-iE_kt}$ → reconstruct against $e^{+iEt}$ or
   every energy flips sign.
8. **Peak-ratio labels blow up as $p_k\to0$.** Report labels only above a stated weight threshold
   and quantify their uncertainty.
9. **Never compare methods across different shot budgets** — the efficiency score assumes the
   fixed totals of §0.2.
10. **Rerun from a clean kernel before submitting.** Frozen seeds, pinned versions, saved figures.
11. **Aer `save_*` with `conditional=True` gets reordered by `transpile()`** — insert a
    `qc.barrier()` (§3.4).
12. **$\langle Z_0\rangle$ is not $\langle Z_0\rangle_\rho$.** Unweighted shadows of a
    non-conserved observable measure $\operatorname{Tr}[O\rho^{(I)}(t)]$. Comparing against the
    input-state value will look like a bias and is not.

## 10.4 References

1. P. K. Faehrmann, J. Eisert, R. Kueng, *In the Shadow of the Hadamard Test: Using the Garbage
   State for Good and Further Modifications*, **Phys. Rev. Lett. 135, 150603 (2025)**;
   [arXiv:2505.15913](https://arxiv.org/abs/2505.15913).
2. H.-Y. Huang, R. Kueng, J. Preskill, *Predicting Many Properties of a Quantum System from Very
   Few Measurements*, **Nature Physics 16, 1050 (2020)**.
3. L. Lin, Y. Tong, *Heisenberg-Limited Ground-State Energy Estimation for Early Fault-Tolerant
   Quantum Computers*, **PRX Quantum 3, 010318 (2022)**.
4. K. Wan, M. Berta, E. T. Campbell, *Randomized Quantum Algorithm for Statistical Phase
   Estimation*, **Phys. Rev. Lett. 129, 030503 (2022)**.
5. K. Klymko *et al.*, *Real-Time Evolution for Ultracompact Hamiltonian Eigenstates on Quantum
   Hardware*, **PRX Quantum 3, 020323 (2022)**.
6. S. Scali, J. Kirsopp, A. Márquez Romero, M. Krompiec, *Purified Phase Estimation Samples Spectra
   Efficiently*, [arXiv:2510.14744](https://arxiv.org/abs/2510.14744).
7. Qiskit, `PauliEvolutionGate` API documentation —
   <https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.PauliEvolutionGate>
8. Qiskit Aer, `AerSimulator` API documentation —
   <https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html>
9. `povm-toolbox` (qiskit-community), classical shadows for Qiskit —
   <https://github.com/qiskit-community/povm-toolbox>

---

*Good luck. The garbage register is worth as much as the ancilla — go prove it.*